# Trade-Based Money Laundering (TBML) Detection Using a Hybrid Statistical and Machine Learning Framework

**Assignment:** ML_01 (Phase 02) — ML-Based Detection of Trade Mis-Invoicing
**Method family:** Hybrid — robust statistical peer-comparison + unsupervised multivariate
anomaly detection (Isolation Forest), combined via a validation-selected weighted
geometric-mean ensemble.

## Executive Summary

This notebook detects suspected over-/under-invoicing in trade transactions and explains
_why_ each transaction is flagged, in language a compliance official — not just a data
scientist — can act on. The methodology is a direct synthesis of two literature threads
introduced in Phase 01: Choi (2019)'s WCO Price Filter Method / Partner Country Method
cross-referencing logic, and the TBML shipping/invoicing typologies documented by
Simmons & Simmons and FATF/OECD. Nitsch (2017)'s central methodological warning —
that trade-misinvoicing point estimates are only meaningful alongside their generating
assumptions — is treated as a design constraint throughout, not an afterthought.

### Pipeline at a Glance

| Stage               | Module(s) | What Happens                                                                                                                                                                   |
| ------------------- | --------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| Data generation     | 3–5       | Synthetic transactions with **literature-derived, mathematically calibrated** risk correlations (country, shell-importer, quantity-direction coupling) — not independent noise |
| Partitioning        | 7         | Stratified 60/20/20 train/validation/test split                                                                                                                                |
| Feature engineering | 9         | Leakage-safe, peer-normalized signals (robust z-score via MAD, log-transforms) computed from train statistics only                                                             |
| Detection           | 10–12     | Two independent detectors (statistical peer-comparison, Isolation Forest) combined via a weighted geometric-mean ensemble whose weights are validation-selected                |
| Decision            | 13–14     | Risk threshold and direction threshold both derived by grid search against validation-set $F_2$ / macro-F1 — never hand-picked                                                 |
| Evaluation          | 15–16     | Held-out test-set metrics, reframed in customs/AML "targeting accuracy" language and explicitly calibrated against Choi (2019)'s real-world benchmark                          |
| Validation          | 17, 21    | Visual diagnostics plus a 5-seed repeated-split robustness check reporting mean ± std, not a single point estimate                                                             |
| Explainability      | 18, 20    | Transaction-level component decomposition and an interactive manual predictor                                                                                                  |
| Honesty check       | 22        | Explicit limitations connecting every simplifying assumption back to the source literature                                                                                     |

### Core Design Principle

Every numeric constant used for a _decision_ (contamination, ensemble weights, risk
threshold, direction threshold, rare-importer cutoff) is **derived by optimizing an
explicit, stated objective against validation data** — never hand-picked. Constants
used to define the _synthetic ground truth itself_ (risk multipliers, price-multiplier
bands) are instead **anchored to cited real-world typology sources**, with their
derivation shown at the point of use (Module 5). Section 22 states plainly where this
rigor still meets a real-world limitation.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Imports
# ═══════════════════════════════════════════════════════════════════════════

import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="talk", palette="deep")

# ───────────────────────────────────────────────────────────────────────────
# Reproducibility
# ───────────────────────────────────────────────────────────────────────────
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Libraries imported successfully.")
print(f"Random seed set to: {RANDOM_SEED}")

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Utility Functions
# ───────────────────────────────────────────────────────────────────────────


def min_max_scale(values):
    """
    Scale an array to the [0, 1] range.
    """
    minimum = values.min()
    maximum = values.max()

    return (values - minimum) / (maximum - minimum + 1e-9)


def min_max_scale_with_bounds(
    values,
    minimum,
    maximum,
):
    """
    Scale using externally supplied min/max values
    (typically learned from the training set).
    """
    return np.clip(
        (values - minimum) / (maximum - minimum + 1e-9),
        0,
        1,
    )

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Dataset Parameters
# ═══════════════════════════════════════════════════════════════════════════

N_TRANSACTIONS = 100_000
ANOMALY_RATE = 0.005

TEST_SIZE = 0.20

OVER_RATIO = 0.60
EXTREME_ANOMALY_FRACTION = 0.40

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Risk-Correlation Constants (mathematically calibrated, not arbitrary)
# ═══════════════════════════════════════════════════════════════════════════
#
# COUNTRY_RISK_MULTIPLIER: FATF/WCO typology reports (WCO, 2018a; APG, 2012)
# repeatedly flag jurisdictions such as Panama, UAE, Cayman Islands, Malta
# and Seychelles as common conduits for trade-based capital flight — encoded
# here as a 4x elevated anomaly probability for HIGH_RISK_COUNTRIES.
COUNTRY_RISK_MULTIPLIER = 4.0

# SHELL_IMPORTER_MULTIPLIER: Simmons & Simmons ("Trade Based Money
# Laundering") document the use of shell/front companies used briefly then
# discarded (U-Boating, sanctions-circumvention fronts) — encoded as a small
# subset of importer IDs with low transaction volume + elevated anomaly risk.
SHELL_IMPORTER_MULTIPLIER = 6.0
SHELL_IMPORTER_SHARE_OF_VOLUME = 0.03  # shell IDs get ~3% of total volume
NUM_NORMAL_IMPORTERS = 250
NUM_SHELL_IMPORTERS = 50

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Quantity-Distortion Constants (Short/Over-Shipping — Simmons & Simmons TBML)
# ═══════════════════════════════════════════════════════════════════════════
#
# Over-invoicing often pairs with SHORT shipping (fewer goods than invoiced,
# to inflate declared value without moving proportionally more product).
# Under-invoicing often pairs with OVER shipping (more goods than invoiced,
# moving product while declaring less value per unit).
SHORT_SHIP_FACTOR_EXTREME = (0.35, 0.65)
SHORT_SHIP_FACTOR_MILD = (0.70, 0.90)
OVER_SHIP_FACTOR_EXTREME = (1.35, 1.90)
OVER_SHIP_FACTOR_MILD = (1.10, 1.30)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Product Master Data
# ═══════════════════════════════════════════════════════════════════════════

PRODUCT_CATEGORIES = {
    "electronics": (300, 600, 0.10),
    "machinery": (800, 2000, 0.10),
    "medical_devices": (400, 1200, 0.10),
    "automobile_parts": (120, 700, 0.11),
    "industrial_tools": (80, 500, 0.11),
    "textile": (8, 20, 0.12),
    "garments": (10, 60, 0.14),
    "footwear": (15, 80, 0.13),
    "chemicals": (60, 180, 0.12),
    "pharmaceuticals": (80, 500, 0.12),
    "cosmetics": (20, 150, 0.13),
    "food_products": (2, 8, 0.15),
    "agricultural_products": (3, 25, 0.15),
    "seafood": (8, 40, 0.14),
    "raw_materials": (20, 60, 0.10),
    "steel_products": (40, 180, 0.10),
    "plastic_products": (15, 70, 0.11),
    "rubber_products": (20, 90, 0.11),
    "furniture": (100, 400, 0.12),
    "paper_products": (5, 30, 0.13),
}

HS_CODE_MAP = {
    "electronics": "851712",
    "machinery": "847989",
    "medical_devices": "901890",
    "automobile_parts": "870899",
    "industrial_tools": "820559",
    "textile": "520942",
    "garments": "610910",
    "footwear": "640399",
    "chemicals": "280110",
    "pharmaceuticals": "300490",
    "cosmetics": "330499",
    "food_products": "100630",
    "agricultural_products": "100199",
    "seafood": "030617",
    "raw_materials": "720839",
    "steel_products": "721049",
    "plastic_products": "392690",
    "rubber_products": "401699",
    "furniture": "940360",
    "paper_products": "481910",
}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Country Master Data
# ═══════════════════════════════════════════════════════════════════════════

HIGH_RISK_COUNTRIES = [
    "Panama",
    "UAE",
    "Cayman Islands",
    "Malta",
    "Seychelles",
]

LOW_RISK_COUNTRIES = [
    "Germany",
    "Japan",
    "UK",
    "Singapore",
    "Canada",
    "Australia",
    "China",
    "India",
    "Vietnam",
    "Turkey",
]

ALL_COUNTRIES = LOW_RISK_COUNTRIES + HIGH_RISK_COUNTRIES

In [ ]:
def print_dataset_configuration():
    print(f"Products         : {len(PRODUCT_CATEGORIES)}")
    print(
        f"Countries        : {len(ALL_COUNTRIES)} "
        f"({len(HIGH_RISK_COUNTRIES)} high-risk)"
    )
    print(f"Target size      : {N_TRANSACTIONS:,} transactions")
    print(f"Max anomaly rate : {ANOMALY_RATE:.2%}")


print_dataset_configuration()

## 5. Synthetic Data Generation Model

### 5.1 Generative Model Overview

Each transaction $i$ is generated as a random draw from a joint distribution:

$$
(\text{category}_i,\ \text{quantity}_i,\ \text{origin}_i,\ \text{importer}_i,\ \text{anomaly}_i,\ \text{price}_i)
$$

where the anomaly indicator $A_i \in \{0, 1\}$ is drawn **first**, and every downstream
quantity (price, shipped quantity) is conditioned on $A_i$. This ordering is intentional:
it lets us inject literature-grounded _risk correlations_ (Section 5.2) instead of treating
country, importer, and quantity as independent of fraud status — a simplification that
would make every feature except price uninformative by construction.

### 5.2 Risk-Adjusted Anomaly Probability

For transaction $i$ with high-risk-country indicator $h_i \in \{0,1\}$ and
shell-importer indicator $s_i \in \{0,1\}$, the anomaly probability is:

$$
p_i = \min\Big(p_0 \cdot m_i,\ 0.35\Big), \qquad
m_i = \underbrace{r_c^{\,h_i}}_{\text{country multiplier}} \cdot \underbrace{r_s^{\,s_i}}_{\text{importer multiplier}}
$$

where $r_c = \text{COUNTRY\_RISK\_MULTIPLIER} = 4.0$ and
$r_s = \text{SHELL\_IMPORTER\_MULTIPLIER} = 6.0$ are literature-grounded constants
(FATF/OECD 2006; WCO 2018a; Simmons & Simmons — shell/front-company typology), **not**
arbitrary tuning knobs. The 0.35 cap prevents degenerate near-certain anomaly rates for
transactions that are both high-risk-country _and_ shell-importer.

### 5.3 Deriving the Base Rate $p_0$ (not assumed — solved for)

We require the **population-average** anomaly rate to equal the target
$\text{ANOMALY\_RATE} = 0.005$, _after_ the multipliers above are applied. Taking the
expectation of $p_i$ over the population distribution of $h_i$ and $s_i$:

$$
\mathbb{E}[A] = p_0 \cdot \mathbb{E}[m_i] = p_0 \cdot \mathbb{E}[r_c^{\,h}] \cdot \mathbb{E}[r_s^{\,s}]
$$

Since $h_i \sim \text{Bernoulli}(f_{hr})$ and $s_i \sim \text{Bernoulli}(f_{shell})$ with
known population shares $f_{hr} = |\text{HIGH\_RISK\_COUNTRIES}| / |\text{ALL\_COUNTRIES}|$
and $f_{shell} = \text{SHELL\_IMPORTER\_SHARE\_OF\_VOLUME}$:

$$
\mathbb{E}[r_c^{\,h}] = f_{hr}\, r_c + (1-f_{hr}) \cdot 1
\qquad
\mathbb{E}[r_s^{\,s}] = f_{shell}\, r_s + (1-f_{shell}) \cdot 1
$$

Solving for the base rate that makes $\mathbb{E}[A] = \text{ANOMALY\_RATE}$:

$$
\boxed{p_0 = \dfrac{\text{ANOMALY\_RATE}}{\mathbb{E}[r_c^{\,h}] \cdot \mathbb{E}[r_s^{\,s}]}}
$$

This is exactly what `base_rate = ANOMALY_RATE / (e_m_country * e_m_importer)` computes
in code — **the constant is derived from the target rate and known population shares,
not hand-picked.** The empirical check (`print_dataset_summary`) confirms the realized
ratios (≈4.2× and ≈5.8×) match the design targets (4.0×, 6.0×) within sampling noise.

### 5.4 Price Multiplier Ranges — Empirically Anchored

For an anomalous transaction, the unit price is set to a multiple of the category's
normal price band $[lo, hi]$:

$$
\text{price}_i =
\begin{cases}
hi \cdot U(3.0, 8.0) & \text{extreme over-invoicing} \\
hi \cdot U(1.15, 1.5) & \text{mild over-invoicing} \\
lo \cdot U(0.05, 0.30) & \text{extreme under-invoicing} \\
lo \cdot U(0.60, 0.85) & \text{mild under-invoicing}
\end{cases}
$$

These ranges are **anchored to documented real-world cases** (Simmons & Simmons, _Trade
Based Money Laundering_): toilet tissue overinvoiced at US\$4,121/kg, sugar overinvoiced
at US\$240/lb against a normal US\$0.10–0.30/lb (≈800–2400×). Real cases can therefore be
two to three orders of magnitude more extreme than the 3–8× band used here; the band is
deliberately conservative so the detection problem remains realistically hard rather than
trivial, while still being far less arbitrary than an unmotivated 1.8–2.8× band would be.

### 5.5 Quantity Distortion — Coupled to Invoicing Direction

Per the shipping-technique typology (Simmons & Simmons — short/over shipping), quantity
is **not independent** of the price anomaly direction:

$$
\text{qty}_i = \text{qty}_i^{\text{base}} \times
\begin{cases}
U(0.35, 0.65) \text{ or } U(0.70, 0.90) & \text{if OVER\_INVOICING (short-shipping)}\\
U(1.35, 1.90) \text{ or } U(1.10, 1.30) & \text{if UNDER\_INVOICING (over-shipping)}
\end{cases}
$$

This encodes the economic logic: over-invoicing inflates declared value without moving
proportionally more product (fewer goods, higher declared price), while under-invoicing
moves more product while declaring less value per unit.


In [ ]:
# ───────────────────────────────────────────────────────────────────────────
#  Helper: Quantity Generator
# ───────────────────────────────────────────────────────────────────────────
def generate_quantity(rng):
    """Generate realistic trade quantity with lognormal distribution."""
    return max(1, min(int(rng.lognormal(4.0, 0.8)), 5000))


# ───────────────────────────────────────────────────────────────────────────
#  Helper: Quantity Distortion for Anomalous Transactions
# ───────────────────────────────────────────────────────────────────────────
def distort_quantity(rng, base_qty, direction, is_extreme):
    """
    Apply short-/over-shipping distortion to quantity based on invoicing
    direction (Simmons & Simmons TBML shipping-technique typology).
    """
    if direction == "OVER_INVOICING":
        lo_f, hi_f = SHORT_SHIP_FACTOR_EXTREME if is_extreme else SHORT_SHIP_FACTOR_MILD
    else:  # UNDER_INVOICING
        lo_f, hi_f = OVER_SHIP_FACTOR_EXTREME if is_extreme else OVER_SHIP_FACTOR_MILD

    factor = rng.uniform(lo_f, hi_f)
    return max(1, int(round(base_qty * factor)))


# ───────────────────────────────────────────────────────────────────────────
#  Helper: Normal Pricing
# ───────────────────────────────────────────────────────────────────────────
def generate_normal_price(rng, lo, hi, cv):
    """Generate unit price for normal transactions using lognormal distribution."""
    mid = (lo + hi) / 2
    std = mid * cv
    variance = (std / mid) ** 2
    sigma = np.sqrt(np.log(1 + variance))
    mu = np.log(mid) - sigma**2 / 2
    return round(max(0.01, rng.lognormal(mu, sigma)), 2)


# ───────────────────────────────────────────────────────────────────────────
#  Helper: Anomaly Pricing
# ───────────────────────────────────────────────────────────────────────────
def generate_anomaly_price(rng, lo, hi, is_extreme, over_ratio):
    """
    Generate anomalous unit price (over/under invoicing).

    Multiplier ranges informed by documented TBML cases (Simmons & Simmons,
    "Trade Based Money Laundering"): toilet tissue overinvoiced at
    US$4,121/kg; sugar overinvoiced at US$240/lb vs a normal 10-30 cents/lb
    (~800-2400x). Real cases can reach multiples in the hundreds; we use a
    more conservative "extreme" band (3x-8x / 0.05x-0.30x) so the detection
    problem stays realistically hard, while being far less conservative
    than an arbitrary 1.8x-2.8x band would be.
    """
    if rng.random() < over_ratio:
        direction = "OVER_INVOICING"
        multiplier = rng.uniform(3.0, 8.0) if is_extreme else rng.uniform(1.15, 1.5)
        unit_price = hi * multiplier * rng.uniform(0.95, 1.05)
    else:
        direction = "UNDER_INVOICING"
        multiplier = rng.uniform(0.05, 0.30) if is_extreme else rng.uniform(0.60, 0.85)
        unit_price = lo * multiplier * rng.uniform(0.95, 1.05)

    return round(max(0.01, unit_price), 2), direction


# ───────────────────────────────────────────────────────────────────────────
#  Main: Synthetic Dataset Generator
# ───────────────────────────────────────────────────────────────────────────
def generate_synthetic_dataset():
    """Generate the complete synthetic trade dataset with risk-correlated anomalies."""

    # ── Importer pool: 250 "normal" importers carry ~97% of volume;
    #    50 "shell-style" importers carry ~3% of volume (used briefly,
    #    rarely repeated) and carry an elevated anomaly probability.
    normal_importers = np.arange(1, NUM_NORMAL_IMPORTERS + 1)
    shell_importers = np.arange(
        NUM_NORMAL_IMPORTERS + 1, NUM_NORMAL_IMPORTERS + NUM_SHELL_IMPORTERS + 1
    )
    importer_pool = np.concatenate([normal_importers, shell_importers])
    shell_importer_set = set(shell_importers.tolist())

    normal_weight = (1 - SHELL_IMPORTER_SHARE_OF_VOLUME) / NUM_NORMAL_IMPORTERS
    shell_weight = SHELL_IMPORTER_SHARE_OF_VOLUME / NUM_SHELL_IMPORTERS
    importer_weights = np.concatenate(
        [
            np.full(NUM_NORMAL_IMPORTERS, normal_weight),
            np.full(NUM_SHELL_IMPORTERS, shell_weight),
        ]
    )

    # ── Analytically derive BASE_RATE so the overall anomaly rate still
    #    equals ANOMALY_RATE once country-risk and importer-risk
    #    multipliers are applied. E[m] is computed from known population
    #    shares of each risk factor (not an arbitrary guess).
    f_hr = len(HIGH_RISK_COUNTRIES) / len(ALL_COUNTRIES)
    f_shell = SHELL_IMPORTER_SHARE_OF_VOLUME

    e_m_country = f_hr * COUNTRY_RISK_MULTIPLIER + (1 - f_hr) * 1.0
    e_m_importer = f_shell * SHELL_IMPORTER_MULTIPLIER + (1 - f_shell) * 1.0
    base_rate = ANOMALY_RATE / (e_m_country * e_m_importer)

    rows = []

    for i in range(N_TRANSACTIONS):

        # Product category setup
        cat = rng.choice(list(PRODUCT_CATEGORIES))
        lo, hi, cv = PRODUCT_CATEGORIES[cat]

        # Country & importer (weighted so shell importers appear rarely)
        origin = rng.choice(ALL_COUNTRIES)
        hr = int(origin in HIGH_RISK_COUNTRIES)
        importer = int(rng.choice(importer_pool, p=importer_weights))
        is_shell = importer in shell_importer_set

        # Risk-adjusted anomaly probability (derivation above)
        m = (COUNTRY_RISK_MULTIPLIER if hr else 1.0) * (
            SHELL_IMPORTER_MULTIPLIER if is_shell else 1.0
        )
        p_anomaly = min(base_rate * m, 0.35)  # cap to avoid degenerate probabilities

        is_anomaly = int(rng.random() < p_anomaly)

        # Base quantity (before any anomaly-driven shipping distortion)
        qty = generate_quantity(rng)

        if is_anomaly:
            is_extreme = rng.random() < EXTREME_ANOMALY_FRACTION
            unit_price, direction = generate_anomaly_price(
                rng, lo, hi, is_extreme, OVER_RATIO
            )
            qty = distort_quantity(rng, qty, direction, is_extreme)
        else:
            unit_price = generate_normal_price(rng, lo, hi, cv)
            direction = "NORMAL"

        rows.append(
            {
                "txn_id": f"TXN{i+1:05d}",
                "product_group": cat,
                "hs_code": HS_CODE_MAP[cat],
                "quantity": qty,
                "unit_price": unit_price,
                "declared_value": round(unit_price * qty, 2),
                "country_origin": origin,
                "country_risk": hr,
                "importer_id": importer,
                "is_shell_importer": int(is_shell),
                "anomaly_label": is_anomaly,
                "true_direction": direction,
            }
        )

    return pd.DataFrame(rows)


# ───────────────────────────────────────────────────────────────────────────
#  Dataset Summary Printer
# ───────────────────────────────────────────────────────────────────────────
def print_dataset_summary(df):
    """Print summary statistics of the synthetic dataset."""
    y = df["anomaly_label"].values
    total = len(df)
    n_anom = int(y.sum())
    n_normal = total - n_anom

    print("=" * 60)
    print("  Synthetic Dataset Summary")
    print("=" * 60)
    print(f"  Total          : {total:,}")
    print(f"  Normal         : {n_normal:,} ({n_normal/total*100:.1f}%)")
    print(f"  Anomalies      : {n_anom:,} ({n_anom/total*100:.1f}%)")
    print(f"  Over-invoicing : {(df.true_direction=='OVER_INVOICING').sum()}")
    print(f"  Under-invoicing: {(df.true_direction=='UNDER_INVOICING').sum()}")
    print("-" * 60)
    print("  Injected Risk-Correlation Check")
    print("-" * 60)
    print("Anomaly rate by country_risk (0=low, 1=high):")
    print(
        df.groupby("country_risk")["anomaly_label"].mean().mul(100).round(2).to_string()
    )
    print("\nAnomaly rate by is_shell_importer (0=normal, 1=shell):")
    print(
        df.groupby("is_shell_importer")["anomaly_label"]
        .mean()
        .mul(100)
        .round(2)
        .to_string()
    )
    print("\nMean quantity by true_direction (short/over-shipping check):")
    print(df.groupby("true_direction")["quantity"].mean().round(1).to_string())


# ───────────────────────────────────────────────────────────────────────────
#  Execution
# ───────────────────────────────────────────────────────────────────────────
df = generate_synthetic_dataset()
print_dataset_summary(df)

## 6. Dataset Inspection (Sanity Check)

Before any splitting or feature engineering, a quick structural check confirms the
generator (Module 5) produced the expected schema, row count, and value ranges —
catching accidental type errors or malformed rows early.


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

## 7. Train / Validation / Test Partitioning

The dataset is split into a 60/20/20 stratified train/validation/test partition
(stratified on `anomaly_label`, so the ≈0.5% anomaly rate is preserved in all three
splits). Train fits statistics and models (Module 9–11); validation selects every
hyperparameter (Modules 11–14); test is touched exactly once, in Module 15, for final
unbiased evaluation.


In [ ]:
# ───────────────────────────────────────────────────────────────────────────
#  Helper: Perform Train/Validation/Test Split (stratified, 3-way)
# ───────────────────────────────────────────────────────────────────────────
def perform_train_val_test_split(
    df, y, val_size=0.20, test_size=0.20, random_state=RANDOM_SEED
):
    """
    Stratified 3-way split.
    Step 1: carve out TEST from the full data.
    Step 2: carve out VAL from the remaining TRAIN+VAL pool.
    """
    trainval_idx, test_idx = train_test_split(
        df.index,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    y_trainval = y[trainval_idx]
    # val_size given as a fraction of the FULL dataset;
    # convert it to a fraction of the trainval pool
    relative_val_size = val_size / (1 - test_size)

    train_idx, val_idx = train_test_split(
        trainval_idx,
        test_size=relative_val_size,
        stratify=y_trainval,
        random_state=random_state,
    )

    df = df.copy()
    df["split"] = "test"
    df.loc[train_idx, "split"] = "train"
    df.loc[val_idx, "split"] = "val"

    train_mask = (df["split"] == "train").values
    val_mask = (df["split"] == "val").values
    test_mask = (df["split"] == "test").values

    return df, train_mask, val_mask, test_mask


# ───────────────────────────────────────────────────────────────────────────
#  Helper: Print Split Summary
# ───────────────────────────────────────────────────────────────────────────
def print_split_summary(y, train_mask, val_mask, test_mask):
    print("=" * 60)
    print("Train / Validation / Test Split Summary")
    print("=" * 60)
    print(f"Train size : {train_mask.sum():,} (anomalies: {y[train_mask].sum()})")
    print(f"Val size   : {val_mask.sum():,} (anomalies: {y[val_mask].sum()})")
    print(f"Test size  : {test_mask.sum():,} (anomalies: {y[test_mask].sum()})")


# ───────────────────────────────────────────────────────────────────────────
#  Execution
# ───────────────────────────────────────────────────────────────────────────
VAL_SIZE = 0.20
TEST_SIZE = 0.20  # keep your existing TEST_SIZE variable name/value

y = df["anomaly_label"].values
df, train_mask, val_mask, test_mask = perform_train_val_test_split(
    df, y, val_size=VAL_SIZE, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
print_split_summary(y, train_mask, val_mask, test_mask)

## 8. Exploratory Data Analysis

These four views confirm the generative model (Module 5) behaves as specified before
any modeling begins: the realized class imbalance (EDA1), the product category mix
(EDA2), fraud rate by category — which should show no strong category-specific skew
since `ANOMALY_RATE` and the risk multipliers are category-agnostic by design (EDA3),
and the unit-price distribution split by class, an early visual preview of the
separability quantified later in Module 9–10 (EDA4).


In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# EDA 1 — Dataset Class Distribution
# ───────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
plot = sns.countplot(
    data=df,
    x="anomaly_label",
    palette="Set2",
    width=0.55,
    edgecolor="black",
    linewidth=1.2,
)
plot.set_xticklabels(["Normal", "Anomaly"])
plt.title("Dataset Class Distribution", fontsize=18, weight="bold", pad=15)
plt.xlabel("Transaction Class", fontsize=13)
plt.ylabel("Number of Transactions", fontsize=13)

total_n = len(df)
for p in plot.patches:
    count = int(p.get_height())
    percent = 100 * count / total_n
    plot.annotate(
        f"{count:,}\n({percent:.1f}%)",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        xytext=(0, 6),
        textcoords="offset points",
    )

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# EDA 2 — Product Category Distribution
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
ax = sns.countplot(
    data=df,
    y="product_group",
    order=df["product_group"].value_counts().index,
    palette="viridis",
    edgecolor="black",
    linewidth=1.1,
)
plt.title("Transaction Distribution by Product Category", weight="bold", pad=15)
plt.xlabel(
    "Number of Transactions",
)
plt.ylabel(
    "Product Category",
)

for p in ax.patches:
    width = int(p.get_width())
    ax.annotate(
        f"{width:,}",
        (width, p.get_y() + p.get_height() / 2),
        xytext=(6, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontweight="bold",
    )

sns.despine(left=False)
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# EDA 3 — Fraud Rate by Product Category
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
fraud_rate = (
    df.groupby("product_group")["anomaly_label"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .reset_index(name="fraud_rate")
)

ax = sns.barplot(
    data=fraud_rate,
    y="product_group",
    x="fraud_rate",
    palette="magma",
    edgecolor="black",
    linewidth=1.1,
)
plt.title("Fraud Rate by Product Category", fontsize=18, weight="bold", pad=15)
plt.xlabel("Fraud Rate (%)", fontsize=13)
plt.ylabel("Product Category", fontsize=13)

for p in ax.patches:
    value = p.get_width()
    ax.annotate(
        f"{value:.2f}%",
        (value, p.get_y() + p.get_height() / 2),
        xytext=(6, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=11,
        fontweight="bold",
    )

ax.set_xlim(0, fraud_rate["fraud_rate"].max() * 1.15)
sns.despine(left=False)
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# EDA 4 — Unit Price Distribution by Class
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
sns.histplot(
    data=df,
    x="unit_price",
    hue="anomaly_label",
    bins=60,
    stat="density",
    common_norm=False,
    kde=True,
    palette=["#4C72B0", "#DD8452"],
    alpha=0.55,
    edgecolor=None,
)
plt.title("Distribution of Unit Price", fontsize=18, fontweight="bold", pad=15)
plt.xlabel("Unit Price", fontsize=13)
plt.ylabel("Density", fontsize=13)
plt.legend(title="Transaction Type", labels=["Normal", "Anomaly"], frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

## 9. Feature Engineering: Leakage-Safe, Peer-Normalized Signals

### 9.1 The Leakage-Safety Constraint

Every statistic below (medians, MADs, means, standard deviations, importer frequencies)
is computed **exclusively from `train_df`**, then broadcast onto train, validation, and
test rows via lookup/transform. This is not optional bookkeeping: any statistic computed
on the full `df` (including val/test rows) would let information about held-out anomalies
leak into the normalization itself, silently inflating downstream validation and test
metrics. Formally, for any statistic $\hat\theta$ used to construct a feature $f(x_i;\hat\theta)$:

$$
\hat\theta = \hat\theta(\{x_j : j \in \text{train}\}) \quad \text{never} \quad \hat\theta(\{x_j : j \in \text{train} \cup \text{val} \cup \text{test}\})
$$

### 9.2 Robust Z-score via Median Absolute Deviation (MAD)

For each HS code group $g$, the ordinary (mean/std) z-score is not used because unit
prices are heavy-tailed and a single extreme value can distort $\mu_g$ and $\sigma_g$
themselves — precisely the failure mode Choi (2019) and Nitsch (2017) warn about when
using naive statistical outlier rules on trade data. Instead we use the **median** and
**median absolute deviation**, which have a breakdown point of 50% (i.e. up to half the
group's data can be corrupted before the estimator itself becomes unreliable):

$$
\widetilde{p}_g = \operatorname{median}\big(\{p_j : j \in \text{train}, \text{hs\_code}_j = g\}\big)
$$

$$
\text{MAD}_g = \operatorname{median}\Big(\big|\,p_j - \widetilde{p}_g\,\big| : j \in \text{train}, \text{hs\_code}_j = g\Big)
$$

The robust z-score for any transaction $i$ in group $g$ is then:

$$
z_i = \frac{p_i - \widetilde{p}_g}{\text{MAD}_g + \varepsilon}, \qquad \varepsilon = 10^{-9} \text{ (guards divide-by-zero)}
$$

This is the direct analogue of Choi (2019)'s Price Filter Method (PFM) unit-price
analysis, replacing PFM's interquartile bounds with a MAD-based estimator for
robustness — both formalize the same idea: _flag transactions whose price is
statistically far from their peer group's central tendency._

### 9.3 Price Ratio and its Log-Transform

A second, complementary view of the same deviation is the raw ratio and its
log-transform:

$$
r_i = \frac{p_i}{\widetilde{p}_g + \varepsilon}, \qquad
\ell_i = \log(1 + |r_i - 1|)
$$

The log-transform compresses the right-skewed tail of $r_i$ (a 10× over-invoice and a
0.1× under-invoice are symmetric distortions but $r_i$ treats them asymmetrically;
$\ell_i$ restores approximate symmetry), which stabilizes the min-max scaling applied
in Module 10.

### 9.4 Value and Quantity Anomaly Scores

Declared value $v_i = p_i \times q_i$ is standardized against train-only moments:

$$
z^{(v)}_i = \frac{v_i - \bar v_{\text{train}}}{s_{v,\text{train}}}
$$

Quantity is log-transformed before standardization, since raw trade quantities are
right-skewed (a lognormal generative process, Section 5):

$$
z^{(q)}_i = \frac{\log(1+q_i) - \overline{\log(1+q)}_{\text{train}}}{s_{\log(1+q),\text{train}}}
$$

### 9.5 Importer Frequency — Log-Transform Rationale

Raw per-importer transaction counts are also right-skewed by design (normal importers
carry $\approx 97\%$ of volume across 250 IDs, shell importers $\approx 3\%$ across 50
IDs — see Section 5.2), so the same log-transform is applied for consistency with
quantity:

$$
f_i = \text{count}(\text{importer}_i \text{ in train}), \qquad
\tilde f_i = \log(1 + f_i)
$$

$\tilde f_i$ (not $f_i$) is the feature fed to the Isolation Forest. The **untransformed**
$f_i$ is retained separately for human-readable business explanations, and its
**rare-importer threshold** is derived — not guessed — as the empirical 5th percentile
of the train-set frequency distribution:

$$
\tau_{\text{rare}} = Q_{0.05}\big(\{f_j : j \in \text{train}\}\big)
$$

### 9.6 Why Raw Price Level Is Excluded From the Feature Set

An earlier iteration included raw `unit_price` and `peer_median` directly as model
inputs. Both are dropped here: since `StandardScaler` is fit globally (not per
product category), and unit prices range from \$2 (food) to \$2000 (machinery), a
globally-scaled raw price would let the Isolation Forest split on _absolute price
level_ — which correlates with product category, not with peer-deviation — defeating
the purpose of peer normalization in Section 9.2. Only the already-normalized signals
($z_i$, $z^{(v)}_i$, $z^{(q)}_i$, $\tilde f_i$, country_risk) enter the model.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Feature Engineering (statistics learned from TRAIN split only)
# ═══════════════════════════════════════════════════════════════════════════

train_df = df[df["split"] == "train"]

# Peer median & MAD — computed from train, mapped to everyone
peer_median_map = train_df.groupby("hs_code")["unit_price"].median()

peer_mad_map = train_df.groupby("hs_code")["unit_price"].apply(
    lambda x: (x - x.median()).abs().median()
)

df["peer_median"] = df["hs_code"].map(peer_median_map)
df["MAD"] = df["hs_code"].map(peer_mad_map)

df["robust_z"] = (df["unit_price"] - df["peer_median"]) / (df["MAD"] + 1e-9)
df["abs_robust_z"] = df["robust_z"].abs()

df["price_ratio"] = df["unit_price"] / (df["peer_median"] + 1e-9)
df["log_price_ratio"] = np.log1p(np.abs(df["price_ratio"] - 1))

# Value anomaly — mean/std from train only
value_mean = train_df["declared_value"].mean()
value_std = train_df["declared_value"].std()
df["value_zscore"] = (df["declared_value"] - value_mean) / value_std

# Quantity anomaly — mean/std from train only
df["log_qty"] = np.log1p(df["quantity"])
qty_mean = np.log1p(train_df["quantity"]).mean()
qty_std = np.log1p(train_df["quantity"]).std()
df["qty_zscore"] = (df["log_qty"] - qty_mean) / qty_std

# Importer frequency — counted from train only, log-transformed (raw counts
# are right-skewed: ~310 for normal importers vs ~48 for shell importers)
freq_map = train_df["importer_id"].value_counts()
df["importer_freq"] = df["importer_id"].map(freq_map).fillna(0)
df["log_importer_freq"] = np.log1p(df["importer_freq"])

# Derived (not arbitrary) "rare importer" cutoff: bottom 5th percentile of
# per-importer transaction frequency, computed from TRAIN only. Kept on the
# raw-count scale since it's used for a human-readable business explanation.
RARE_IMPORTER_THRESHOLD = np.percentile(freq_map.values, 5)
print(
    f"Rare-importer threshold (5th pct of train importer frequency) : {RARE_IMPORTER_THRESHOLD:.1f}"
)

FEATURES = [
    "robust_z",
    "value_zscore",
    "qty_zscore",
    "log_importer_freq",
    "country_risk",
]

# Scaler fit on TRAIN only, then applied to everyone
scaler = StandardScaler()
X_scaled = np.zeros((len(df), len(FEATURES)))
X_scaled[train_mask] = scaler.fit_transform(df.loc[train_mask, FEATURES].fillna(0))
X_scaled[val_mask] = scaler.transform(df.loc[val_mask, FEATURES].fillna(0))
X_scaled[test_mask] = scaler.transform(df.loc[test_mask, FEATURES].fillna(0))

print(f"Features engineered : {len(FEATURES)}")
print(f"Feature list         : {FEATURES}")

## 10. Detector A — Statistical Robust Z-Score

With $z_i$ and $\ell_i$ already derived in Module 9.2–9.3, this step only min-max
scales them into $[0,1]$ anomaly scores usable by the ensemble (Module 12):

$$
s_{\text{stat},i} = \text{min\_max\_scale}(|z_i|), \qquad
s_{\text{ratio},i} = \text{min\_max\_scale}(\ell_i)
$$

These are the two price-based arms of the hybrid ensemble; Module 11 adds the third,
multivariate arm.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Statistical Detector — Robust Z-score
# ═══════════════════════════════════════════════════════════════════════════

stat_score = min_max_scale(df["abs_robust_z"].values)

# Price ratio signal — a secondary, complementary price-deviation signal
ratio_signal = min_max_scale(df["log_price_ratio"].values)

print(f"stat_score   range : [{stat_score.min():.3f}, {stat_score.max():.3f}]")
print(f"ratio_signal range : [{ratio_signal.min():.3f}, {ratio_signal.max():.3f}]")

## 11. Detector B — Isolation Forest: Contamination Selection as Optimization

### 11.1 Why a Multivariate Detector Is Needed

The robust z-score (Module 9–10) only sees price deviation. A transaction with normal
price but an unusual _combination_ of quantity, declared value, importer rarity, and
country risk would be invisible to a purely univariate detector. Isolation Forest is
used here as a **multivariate anomaly detector**: it isolates points via random
recursive partitioning, and points requiring fewer splits to isolate (shorter average
path length) are scored as more anomalous.

### 11.2 Contamination as a Hyperparameter, Not a Guess

Isolation Forest requires a `contamination` parameter $c \in (0, 0.5)$, which
determines the internal score-to-label cutoff during training. Rather than assume a
value, it is **selected by grid search against a held-out criterion**:

$$
c^\star = \operatorname*{argmax}_{c \,\in\, \mathcal{C}} \ \text{AP}\Big(y_{\text{val}},\ s_{\text{iso}}(X_{\text{val}}; c)\Big)
$$

where $\mathcal{C} = \{0.005, 0.010, \dots, 0.050\}$, $\text{AP}(\cdot)$ is Average
Precision (area under the precision-recall curve — appropriate here because the
positive class is rare, $\approx 0.5\%$, and AP is far less optimistic than ROC-AUC
under class imbalance), and $s_{\text{iso}}(X;c)$ is the min-max-scaled anomaly score
produced by a forest **fit on train only** with contamination $c$:

$$
s_{\text{iso}}(x) = \text{min\_max\_scale}\big(-\hat h(x)\big), \qquad \hat h(x) = \text{decision\_function}(x)
$$

(the negative sign is needed because scikit-learn's `decision_function` is _higher for
normal points_; negating and rescaling turns it into a conventional 0–1 anomaly score
where higher = more anomalous).

### 11.3 Train/Validation Separation in the Search

Critically, the forest is **fit** on `X_scaled[train_mask]` but **selected** using AP
computed on `X_scaled[val_mask]`. This mirrors standard hyperparameter-tuning practice:
train data determines the model's parameters, validation data determines which
hyperparameter configuration generalizes best, and test data (Module 15) remains
completely unused until final, one-shot evaluation.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Isolation Forest — Contamination Selected via Train-Only Average Precision
# ═══════════════════════════════════════════════════════════════════════════

CONTAMINATION_GRID = np.arange(0.005, 0.055, 0.005)

best_ap_iso, best_contam, best_iso_forest = -1.0, None, None
iso_score = None

for c_val in CONTAMINATION_GRID:
    candidate_forest = IsolationForest(
        n_estimators=300,
        contamination=c_val,
        max_samples="auto",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    candidate_forest.fit(X_scaled[train_mask])
    candidate_score = min_max_scale(-candidate_forest.decision_function(X_scaled))

    ap_val = average_precision_score(y[val_mask], candidate_score[val_mask])

    if ap_val > best_ap_iso:
        best_ap_iso = ap_val
        best_contam = c_val
        best_iso_forest = candidate_forest
        iso_score = candidate_score

print(f"Selected contamination : {best_contam:.3f}  (val AP = {best_ap_iso:.4f})")

# Save train-derived normalization ranges (used later for the interactive
# predictor in the Appendix, so unseen inputs are normalized consistently)
STAT_RAW_MIN = df.loc[train_mask, "abs_robust_z"].min()
STAT_RAW_MAX = df.loc[train_mask, "abs_robust_z"].max()

ISO_RAW_TRAIN = -best_iso_forest.decision_function(X_scaled[train_mask])
ISO_RAW_MIN = ISO_RAW_TRAIN.min()
ISO_RAW_MAX = ISO_RAW_TRAIN.max()

RATIO_RAW_MIN = df.loc[train_mask, "log_price_ratio"].min()
RATIO_RAW_MAX = df.loc[train_mask, "log_price_ratio"].max()

## 12. Hybrid Ensemble Construction

### 12.1 Why a Product, Not a Sum

Given three independent anomaly signals — the statistical detector $s_{\text{stat}}$
(Module 10, from the robust z-score), the multivariate detector $s_{\text{iso}}$
(Module 11), and the price-ratio signal $s_{\text{ratio}}$ (Module 10) — the ensemble
combines them as a **weighted geometric mean**:

$$
E_i = s_{\text{stat},i}^{\,a} \cdot s_{\text{iso},i}^{\,b} \cdot s_{\text{ratio},i}^{\,c}, \qquad a+b+c=1,\ \ a,b,c \ge 0
$$

This is a deliberate choice over a weighted **arithmetic** mean
($E_i = a\,s_{\text{stat}} + b\,s_{\text{iso}} + c\,s_{\text{ratio}}$). Under the
geometric mean, if _any one_ component score is near zero, $E_i$ is pulled toward zero
regardless of how high the other components are — i.e. the ensemble requires
**multiple detectors to agree** before flagging a transaction. This directly mirrors
the cross-referencing logic of Choi (2019)'s PFM $\cap$ PCM methodology, where a
transaction is only treated as highly suspicious when _both_ the price-filter method
and the partner-country method independently flag it. Here, since a second independent
data source (mirror trade data) is not available in the synthetic single-source
setting, the geometric-mean combination of three _independently derived_ signals from
the same data plays an analogous false-positive-reducing role.

### 12.2 The Weight Simplex and Its Optimization

The weights $(a, b, c)$ lie on the 2-simplex $\{(a,b,c) : a+b+c=1,\ a,b,c\ge 0\}$. This
is searched on a grid with step size $0.05$:

$$
(a^\star, b^\star, c^\star) = \operatorname*{argmax}_{(a,b,c)\,\in\,\Delta_{0.05}} \ \text{AP}\Big(y_{\text{val}},\ \text{min\_max\_scale}\big(E(a,b,c)\big)_{\text{val}}\Big)
$$

Both the Isolation Forest fit (Module 11) and this weight search are chained: the
$s_{\text{iso}}$ used here is the one produced by $c^\star_{\text{iso}}$ from Module 11,
and the resulting $(a^\star, b^\star, c^\star)$ is _itself_ validation-selected — no
part of the ensemble's construction touches the test set before Module 15.

### 12.3 Interpreting a Weight of Zero

Because $x^0 = 1$ for any $x > 0$, setting e.g. $b=0$ does not "remove" the Isolation
Forest term — it neutralizes its multiplicative contribution to exactly 1, leaving the
other two terms to determine $E_i$. This is the correct behavior for a weighted
geometric mean and should not be misread as feature deletion.

### 12.4 Why $b^\star$ Is Often Near Zero in Practice

Empirically, the grid search frequently selects $b^\star \approx 0$ (Isolation Forest
receiving little to no ensemble weight), with most of the mass on $a$ (statistical
detector) and $c$ (price-ratio signal). This is a direct, explainable consequence of
Module 5's generative model rather than a flaw in the search: since injected anomalies
are driven predominantly by extreme price multipliers (Module 5.4), and only weakly by
multivariate patterns (quantity, importer rarity, country risk — see Diagnostic 7's
comparatively small and noisier permutation importances for `qty_zscore`,
`log_importer_freq`, `country_risk`), the price-based signals ($s_{\text{stat}}$,
$s_{\text{ratio}}$) already separate the classes almost perfectly on their own.

Because the ensemble is a **geometric mean**, giving $s_{\text{iso}}$ non-trivial weight
would multiplicatively _penalize_ true anomalies whose Isolation Forest score happens to
be only moderately elevated (since their multivariate pattern, while present, is not as
extreme as their price deviation) — lowering validation AP rather than raising it. The
grid search correctly detects this and suppresses $b$.

This should **not** be read as evidence that Isolation Forest, or the multivariate
features it consumes, are useless: Diagnostic 7 shows `country_risk` and
`log_importer_freq` retain positive, non-zero permutation importance _within_ the
forest itself. Rather, it reflects that in _this_ synthetic dataset, price deviation is
the dominant fraud signal by construction (Limitation 22.2) — on real trade data, where
laundering schemes can move genuinely small, price-normal shipments through many rare
shell-importer transactions, the multivariate detector's weight would be expected to
rise, and the ensemble's validation-driven weight selection (Module 12.2) would adapt
to that shift automatically without any code change.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Ensemble Weights — learned from TRAIN-only Average Precision
# ═══════════════════════════════════════════════════════════════════════════

WEIGHT_STEP = 0.05
best_ap, best_weights = -1.0, None
ensemble_score = None

for a in np.arange(0.0, 1.0 + WEIGHT_STEP, WEIGHT_STEP):
    for b in np.arange(0.0, 1.0 - a + WEIGHT_STEP, WEIGHT_STEP):
        c = round(1.0 - a - b, 4)
        if c < 0:
            continue

        candidate_score = (stat_score**a) * (iso_score**b) * (ratio_signal**c)
        candidate_score = min_max_scale(candidate_score)

        ap_val = average_precision_score(y[val_mask], candidate_score[val_mask])

        if ap_val > best_ap:
            best_ap = ap_val
            best_weights = (round(a, 2), round(b, 2), c)
            ensemble_score = candidate_score

a_opt, b_opt, c_opt = best_weights
print(f"Learned weights (val AP = {best_ap:.4f}):")
print(f"  stat exponent  (a) = {a_opt}")
print(f"  iso exponent   (b) = {b_opt}")
print(f"  ratio exponent (c) = {c_opt}")

## 13. Decision Threshold Selection

### 13.1 The Precision-Recall Trade-off, Formalized

Given the ensemble score $E_i \in [0,1]$ (Module 12), a binary flag requires a
threshold $\theta$: $\hat y_i = \mathbb{1}[E_i \ge \theta]$. The choice of $\theta$
trades precision against recall. The $F_\beta$ score formalizes _how much_ recall
should be weighted relative to precision:

$$
F_\beta = (1+\beta^2)\cdot \frac{\text{Precision}\cdot\text{Recall}}{\beta^2\cdot\text{Precision}+\text{Recall}}
$$

At $\beta=1$, precision and recall are weighted equally. At $\beta=2$, recall is
weighted **four times** as heavily as precision (since $\beta^2=4$ appears as the
coefficient on precision's denominator term, effectively discounting precision's
influence on the harmonic mean).

### 13.2 Why $\beta=2$ Is the Correct Choice Here, Not an Arbitrary One

In the AML/TBML domain, a missed fraudulent transaction (false negative) represents a
laundering channel that goes completely undetected, while a false positive only costs
one additional manual review. This asymmetry is exactly what the assignment brief
flags when it warns that accuracy alone is misleading for rare-event detection; $F_2$
operationalizes that warning as a concrete, defensible optimization target rather than
leaving the precision/recall trade-off to informal judgment.

### 13.3 Threshold as a Validation-Selected Percentile

Rather than searching over raw score values directly (which would be sensitive to the
arbitrary scale of $E_i$), the threshold is searched over **percentiles of the
validation-set score distribution**:

$$
\theta^\star = \operatorname*{argmax}_{q \,\in\, \{85.0,\, 85.1,\, \dots,\, 99.4\}} \ F_2\Big(y_{\text{val}},\ \mathbb{1}[E_{\text{val}} \ge \text{Percentile}(E_{\text{val}}, q)]\Big)
$$

The grid's lower bound (85th percentile, versus an initial 93rd percentile in an
earlier iteration) was widened specifically because $F_2$ is recall-favoring and can
select a looser threshold than $F_1$ would; restricting the search to high percentiles
only would bias the result toward higher thresholds than the true $F_2$-optimum.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Decision Threshold — F2-maximizing percentile, VAL-only
#  (F2 weights recall 4x more than precision — appropriate for AML/TBML,
#  where missing real fraud is costlier than an extra manual review)
# ═══════════════════════════════════════════════════════════════════════════

best_f2, best_thr = 0.0, 0.0
for pct in np.arange(85.0, 99.5, 0.1):
    thr = np.percentile(ensemble_score[val_mask], pct)
    pred_val = (ensemble_score[val_mask] >= thr).astype(int)
    f2 = fbeta_score(y[val_mask], pred_val, beta=2, zero_division=0)
    if f2 > best_f2:
        best_f2, best_thr = f2, thr

pred = (ensemble_score >= best_thr).astype(int)

print(f"Selected decision threshold : {best_thr:.4f}  (val F2 = {best_f2:.4f})")

## 14. Direction Classification Threshold

### 14.1 A Separate Decision Problem

Module 13's threshold $\theta^\star$ answers "is this transaction anomalous at all?".
A second, distinct question — "if anomalous, is it over- or under-invoiced?" — is
answered independently, using only the robust z-score $z_i$ (Module 9.2), since
direction is fundamentally a _price-direction_ question rather than a multivariate one:

$$
\widehat{\text{dir}}(z_i) =
\begin{cases}
\text{OVER\_INVOICING} & z_i > \zeta \\
\text{UNDER\_INVOICING} & z_i < -\zeta \\
\text{NORMAL} & |z_i| \le \zeta
\end{cases}
$$

### 14.2 Deriving $\zeta$ via Macro-F1 on Validation

$\zeta$ is chosen, not assumed, by grid search over $\zeta \in \{1.0, 1.1, \dots, 5.0\}$,
maximizing macro-averaged F1 across the three direction classes on the validation
split's _known_ synthetic direction label:

$$
\zeta^\star = \operatorname*{argmax}_{\zeta} \ \text{F1}_{\text{macro}}\Big(\text{true\_direction}_{\text{val}},\ \widehat{\text{dir}}(z_{\text{val}};\, \zeta)\Big)
$$

Macro-averaging (rather than micro/weighted averaging) is used because the three
classes are extremely imbalanced (≈99.5% NORMAL) — a micro-average would be dominated
by the majority class and could report a deceptively high score even if the minority
OVER/UNDER classes were poorly separated.

### 14.3 Consequence: Risk Flag and Direction Can Disagree

Because Module 13's flag and this module's direction use different signals
($E_i$ vs. $z_i$), a transaction can be flagged HIGH risk with `predicted_direction =
NORMAL` — this occurs when the Isolation Forest detects a multivariate anomaly (e.g.
rare importer + high-risk country + quantity anomaly) without an extreme price
deviation. This is handled explicitly in the `explain()` function ("Multivariate
pattern anomalous") and is a feature of the design, not an inconsistency.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Z_THRESHOLD — derived from TRAIN data (direction classification)
# ═══════════════════════════════════════════════════════════════════════════

Z_GRID = np.arange(1.0, 5.05, 0.1)
best_z_f1, Z_THRESHOLD = -1.0, None

true_dir_val = df.loc[val_mask, "true_direction"].values
robust_z_val = df.loc[val_mask, "robust_z"].values

for z_cut in Z_GRID:
    pred_dir = np.where(
        robust_z_val > z_cut,
        "OVER_INVOICING",
        np.where(robust_z_val < -z_cut, "UNDER_INVOICING", "NORMAL"),
    )
    score = f1_score(true_dir_val, pred_dir, average="macro", zero_division=0)
    if score > best_z_f1:
        best_z_f1, Z_THRESHOLD = score, z_cut

print(f"Derived Z_THRESHOLD : {Z_THRESHOLD:.2f}  (val macro-F1 = {best_z_f1:.4f})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Direction Detection & Business Explanation
# ═══════════════════════════════════════════════════════════════════════════
def get_direction(z):
    if z > Z_THRESHOLD:
        return "OVER_INVOICING"
    elif z < -Z_THRESHOLD:
        return "UNDER_INVOICING"
    else:
        return "NORMAL"


df["ensemble_score"] = ensemble_score
df["ensemble_pred"] = pred
df["risk_level"] = pd.Series(pred).map({1: "HIGH", 0: "LOW"}).values
df["predicted_direction"] = df["robust_z"].apply(get_direction)


def explain(row):
    parts = []
    z = row["robust_z"]
    if abs(z) > Z_THRESHOLD:
        side = "above" if z > 0 else "below"
        parts.append(
            f"Unit price {abs(z):.1f}σ {side} peer median "
            f"→ {row['predicted_direction'].replace('_',' ').lower()}"
        )
    if row["ensemble_pred"] == 1 and abs(z) <= Z_THRESHOLD:
        parts.append("Multivariate pattern anomalous (Isolation Forest)")
    if row["country_risk"] == 1:
        parts.append("High-risk trade corridor")
    return " | ".join(parts) if parts else "No significant risk signals"


df["explanation"] = df.apply(explain, axis=1)

print("Direction and explanation columns generated.")
print(
    df[["txn_id", "risk_level", "predicted_direction", "explanation"]]
    .head(3)
    .to_string(index=False)
)

## 15. Model Evaluation — Held-out Test Set

This is the **first and only** point in the pipeline where the test split is used.
Every upstream decision — peer statistics (Module 9), contamination $c^\star$ (Module
11), ensemble weights $(a^\star,b^\star,c^\star)$ (Module 12), decision threshold
$\theta^\star$ (Module 13), direction threshold $\zeta^\star$ (Module 14) — was
selected using only train and validation data. The metrics reported here are therefore
an unbiased estimate of generalization performance, not a number the pipeline was
tuned to produce.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Evaluation Metrics (TEST split only)
# ═══════════════════════════════════════════════════════════════════════════

p = precision_score(y[test_mask], pred[test_mask], zero_division=0)
r = recall_score(y[test_mask], pred[test_mask], zero_division=0)
f1 = f1_score(y[test_mask], pred[test_mask], zero_division=0)
f2 = fbeta_score(y[test_mask], pred[test_mask], beta=2, zero_division=0)
auc = roc_auc_score(y[test_mask], ensemble_score[test_mask])
ap = average_precision_score(y[test_mask], ensemble_score[test_mask])
ba = balanced_accuracy_score(y[test_mask], pred[test_mask])
cm = confusion_matrix(y[test_mask], pred[test_mask])
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn)
fn_r = fn / y[test_mask].sum()

print("=" * 60)
print("  EVALUATION RESULTS (Test Split)")
print("=" * 60)
print(f"  TP = {tp:4d}  (real fraud caught)")
print(f"  FP = {fp:4d}  (false alarms)")
print(f"  TN = {tn:4d}  (normal correctly passed)")
print(f"  FN = {fn:4d}  (fraud missed)")
print("-" * 60)

METRICS = [
    ("Precision", p),
    ("Recall", r),
    ("F1-Score", f1),
    ("F2-Score", f2),
    ("AUC-ROC", auc),
    ("Avg Precision", ap),
    ("Balanced Acc.", ba),
    ("FPR", fpr),
    ("FN rate", fn_r),
]
for name, val in METRICS:
    print(f"  {name:<20} {val*100:>7.2f}%")

print("=" * 60)
print("\nClassification Report:")
print(
    classification_report(
        y[test_mask], pred[test_mask], target_names=["Normal", "Suspicious"], digits=3
    )
)

## 16. Business Targeting-Accuracy Reframing

### 16.1 Precision Restated in Inspection Language

Everything in this section is a **relabeling** of quantities already computed in
Module 15 — no new statistics are introduced. Let $N$ be the test-set size, and let
$F = \{i : \hat y_i = 1\}$ be the flagged set. Then:

$$
\text{Inspection Rate} = \frac{|F|}{N}, \qquad
\text{Targeting Accuracy} = \frac{|F \cap \{i : y_i = 1\}|}{|F|} = \text{Precision}
$$

Targeting accuracy is therefore identical to precision; the relabeling exists because
customs/AML officials reason operationally in terms of _"what fraction of my limited
manual-review capacity gets spent on real fraud"_ rather than in terms of the abstract
precision/recall vocabulary of classification theory.

### 16.2 Lift Over Baseline

The baseline is the population anomaly rate with no model at all,
$\pi = \frac{1}{N}\sum_i y_i$. Lift quantifies how much better than random selection the
model's flagged set is:

$$
\text{Lift} = \frac{\text{Targeting Accuracy}}{\pi}
$$

### 16.3 Calibrating Against Choi (2019)

Choi (2019)'s real-world PFM$\cap$PCM methodology, evaluated on two million actual
customs declarations, achieved a targeting accuracy of 18% against a baseline of 4.5%
— a lift of exactly $18/4.5 = 4\times$. This is the **reference point** against which
this project's own lift figure should be judged, not a target to be matched or
exceeded uncritically: a substantially higher lift on synthetic data is _expected_
given Module 5.4's price-multiplier bands are more separable than real mis-invoicing,
and should be reported as evidence of the synthetic dataset's relative ease, following
Nitsch (2017)'s central caution that point-accuracy claims in this domain are only
meaningful when their generating assumptions are stated alongside them.


In [ ]:
# %%
# ═══════════════════════════════════════════════════════════════════════════
#  Targeting Accuracy Table (Choi, 2019 — WCO methodology framing)
# ═══════════════════════════════════════════════════════════════════════════
#
# Reframes the same test-set results in the language customs/AML officials
# use: how much of the population gets flagged for manual review (inspection
# rate), and among those flagged, what fraction are genuinely anomalous
# (targeting accuracy) — compared against the baseline anomaly rate if no
# model were used at all.

n_test = test_mask.sum()
n_flagged = int(pred[test_mask].sum())
n_flagged_actual_fraud = int(((pred[test_mask] == 1) & (y[test_mask] == 1)).sum())

baseline_rate = y[test_mask].mean() * 100
inspection_rate = n_flagged / n_test * 100
targeting_accuracy = (
    (n_flagged_actual_fraud / n_flagged * 100) if n_flagged > 0 else 0.0
)
lift = targeting_accuracy / baseline_rate if baseline_rate > 0 else float("nan")

targeting_table = pd.DataFrame(
    {
        "Metric": [
            "Total test transactions",
            "Baseline anomaly rate (no model)",
            "Transactions flagged for review (inspection rate)",
            "Flagged transactions that are truly anomalous",
            "Targeting accuracy (precision, business framing)",
            "Lift over baseline",
        ],
        "Value": [
            f"{n_test:,}",
            f"{baseline_rate:.2f}%",
            f"{n_flagged:,} ({inspection_rate:.2f}% of test set)",
            f"{n_flagged_actual_fraud:,}",
            f"{targeting_accuracy:.2f}%",
            f"{lift:.1f}×",
        ],
    }
)

print("=" * 70)
print("  TARGETING ACCURACY SUMMARY (business-facing framing)")
print("=" * 70)
print(targeting_table.to_string(index=False))
print("=" * 70)
print(
    f"\nInterpretation: reviewing only {inspection_rate:.2f}% of transactions "
    f"catches fraud at a rate {lift:.1f}× higher than the baseline "
    f"({baseline_rate:.2f}%). This substantially exceeds Choi (2019)'s reported "
    f"real-world PFM+PCM lift of ~4× (18% targeting accuracy vs a 4.5% baseline). "
    f"The gap is expected: this synthetic dataset's injected anomalies (3-8x / "
    f"0.05-0.30x price multipliers) are more separable than real-world "
    f"mis-invoicing, which is often subtler (see Nitsch, 2017, on the general "
    f"unreliability of point-estimate accuracy claims in this domain). Real-world "
    f"deployment performance should be expected to be lower than this benchmark."
)

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Confusion Matrix
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
cm = confusion_matrix(y[test_mask], pred[test_mask])
labels = np.array(
    [
        [f"True Negative\n{cm[0,0]:,}", f"False Positive\n{cm[0,1]:,}"],
        [f"False Negative\n{cm[1,0]:,}", f"True Positive\n{cm[1,1]:,}"],
    ]
)

sns.heatmap(
    cm,
    annot=labels,
    fmt="",
    cmap="Blues",
    linewidths=1.5,
    linecolor="white",
    square=True,
    cbar=True,
    annot_kws={"fontsize": 12, "fontweight": "bold"},
)
plt.title("Confusion Matrix (Test Split)", fontsize=18, fontweight="bold", pad=15)
plt.xlabel("Predicted Label", fontsize=13, fontweight="bold")
plt.ylabel("Actual Label", fontsize=13, fontweight="bold")
plt.gca().set_xticklabels(["Normal", "Suspicious"], fontsize=12)
plt.gca().set_yticklabels(["Normal", "Suspicious"], fontsize=12, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# ROC Curve
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
fpr_curve, tpr_curve, _ = roc_curve(y[test_mask], ensemble_score[test_mask])

plt.plot(
    fpr_curve,
    tpr_curve,
    color="#1f77b4",
    linewidth=3,
    label=f"Hybrid Ensemble (AUC = {auc:.3f})",
)
plt.fill_between(fpr_curve, tpr_curve, alpha=0.20, color="#1f77b4")
plt.plot(
    [0, 1], [0, 1], linestyle="--", color="gray", linewidth=2, label="Random Classifier"
)
plt.title("ROC Curve (Test Split)", fontsize=18, fontweight="bold", pad=15)
plt.xlabel("False Positive Rate", fontsize=13)
plt.ylabel("True Positive Rate", fontsize=13)
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(alpha=0.30, linestyle="--")
plt.legend(loc="lower right", frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Precision-Recall Curve
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
precision_curve, recall_curve, _ = precision_recall_curve(
    y[test_mask], ensemble_score[test_mask]
)
baseline = y[test_mask].mean()

plt.plot(
    recall_curve,
    precision_curve,
    color="#D62728",
    linewidth=3,
    label=f"Hybrid Ensemble (AP = {ap:.3f})",
)
plt.fill_between(recall_curve, precision_curve, alpha=0.20, color="#D62728")
plt.axhline(
    baseline,
    color="gray",
    linestyle="--",
    linewidth=2,
    label=f"Baseline = {baseline:.3f}",
)
plt.title("Precision–Recall Curve (Test Split)", fontsize=18, fontweight="bold", pad=15)
plt.xlabel("Recall", fontsize=13)
plt.ylabel("Precision", fontsize=13)
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(alpha=0.30, linestyle="--")
plt.legend(loc="lower left", frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Threshold Sensitivity (Test Split)
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
thresholds_plot = np.linspace(0, 1, 200)
precision_list, recall_list, f1_list, f2_list = [], [], [], []

for thr in thresholds_plot:
    preds = (ensemble_score[test_mask] >= thr).astype(int)
    precision_list.append(precision_score(y[test_mask], preds, zero_division=0))
    recall_list.append(recall_score(y[test_mask], preds, zero_division=0))
    f1_list.append(f1_score(y[test_mask], preds, zero_division=0))
    f2_list.append(fbeta_score(y[test_mask], preds, beta=2, zero_division=0))

sns.lineplot(x=thresholds_plot, y=precision_list, linewidth=2.5, label="Precision")
sns.lineplot(x=thresholds_plot, y=recall_list, linewidth=2.5, label="Recall")
sns.lineplot(
    x=thresholds_plot, y=f1_list, linewidth=2.2, linestyle=":", label="F1 Score"
)
sns.lineplot(
    x=thresholds_plot, y=f2_list, linewidth=3.2, label="F2 Score (selection criterion)"
)
plt.axvline(
    best_thr,
    color="black",
    linestyle="--",
    linewidth=2,
    label=f"Selected Threshold = {best_thr:.3f}",
)

best_index = np.argmin(np.abs(thresholds_plot - best_thr))
plt.scatter(
    best_thr, f2_list[best_index], s=140, color="red", edgecolor="black", zorder=5
)

plt.title(
    "Threshold Sensitivity (Test Split) — Selected via F2 on Validation",
    fontsize=18,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Decision Threshold", fontsize=13)
plt.ylabel("Score", fontsize=13)
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(linestyle="--", alpha=0.3)
plt.legend(frameon=True, loc="best")
sns.despine()
plt.tight_layout()
plt.show()

## 17. Diagnostics & Visual Validation

The plots in this section validate — visually — the quantitative claims made in
Modules 9–14: that peer-normalization separates anomalous from normal prices
(Diagnostics 1–2), that the ensemble score concentrates anomalies above the derived
threshold $\theta^\star$ (Diagnostic 3), that the retained feature set is not
internally redundant (Diagnostic 4, correlation matrix), that quantity and value
anomalies co-occur with fraud in the expected direction (Diagnostics 5–6), that
Isolation Forest's feature importances are stable and not artifacts of a single
random split (Diagnostic 7, permutation importance with error bars), and that the
model does not overfit train relative to validation/test (Diagnostic 8).


In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Diagnostic 1 — Unit Price vs Peer Median
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
ax = sns.scatterplot(
    data=df,
    x="peer_median",
    y="unit_price",
    hue="anomaly_label",
    palette=["#4C72B0", "#D62728"],
    alpha=0.70,
    s=45,
    edgecolor="black",
    linewidth=0.3,
)
max_val = max(df["peer_median"].max(), df["unit_price"].max())
plt.plot(
    [0, max_val],
    [0, max_val],
    linestyle="--",
    linewidth=2,
    color="black",
    label="Expected Price",
)
plt.title("Unit Price vs Peer Median", fontsize=18, fontweight="bold", pad=15)
plt.xlabel("Peer Median Unit Price", fontsize=13)
plt.ylabel("Observed Unit Price", fontsize=13)
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ["Normal", "Anomaly"], title="Transaction Type", frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Diagnostic 2 — Robust Z-score Distribution
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
sns.histplot(
    data=df,
    x="robust_z",
    hue="anomaly_label",
    bins=70,
    kde=True,
    stat="density",
    common_norm=False,
    palette=["#4C72B0", "#D62728"],
    alpha=0.55,
    edgecolor=None,
)
plt.axvline(
    x=Z_THRESHOLD,
    color="darkred",
    linestyle="--",
    linewidth=2,
    label=f"+{Z_THRESHOLD:.2f}",
)
plt.axvline(
    x=-Z_THRESHOLD,
    color="darkred",
    linestyle="--",
    linewidth=2,
    label=f"-{Z_THRESHOLD:.2f}",
)
plt.title("Distribution of Robust Z-score", fontsize=18, fontweight="bold", pad=15)
plt.xlabel("Robust Z-score", fontsize=13)
plt.ylabel("Density", fontsize=13)
plt.legend(title="Learned Threshold", frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Diagnostic 3 — Ensemble Score Distribution
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
sns.histplot(
    data=df,
    x="ensemble_score",
    hue="anomaly_label",
    bins=50,
    kde=True,
    stat="density",
    common_norm=False,
    palette=["#4C72B0", "#D62728"],
    alpha=0.60,
    edgecolor=None,
)
plt.axvline(
    best_thr,
    color="black",
    linestyle="--",
    linewidth=2.5,
    label=f"Decision Threshold = {best_thr:.3f}",
)
plt.title(
    "Distribution of Ensemble Anomaly Score", fontsize=18, fontweight="bold", pad=15
)
plt.xlabel("Ensemble Score", fontsize=13)
plt.ylabel("Density", fontsize=13)
plt.xlim(0, 1)
plt.legend(
    title="Model Output",
    labels=["Normal", "Anomaly", f"Threshold = {best_thr:.3f}"],
    frameon=True,
)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Diagnostic 4 — Feature Correlation Heatmap
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
feature_cols_corr = FEATURES + ["ensemble_score"]
corr = df[feature_cols_corr].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.85},
    annot_kws={"size": 9},
)
plt.title("Feature Correlation Matrix", fontsize=18, fontweight="bold", pad=18)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Diagnostic 5 — Declared Value vs Quantity Anomaly
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
ax = sns.scatterplot(
    data=df,
    x="qty_zscore",
    y="value_zscore",
    hue="anomaly_label",
    palette=["#4C72B0", "#D62728"],
    alpha=0.70,
    s=60,
    edgecolor="black",
    linewidth=0.3,
)
plt.axhline(0, linestyle="--", color="gray", linewidth=1.5)
plt.axvline(0, linestyle="--", color="gray", linewidth=1.5)
plt.title("Declared Value vs Quantity Anomaly", fontsize=18, fontweight="bold", pad=15)
plt.xlabel("Quantity Z-score", fontsize=13)
plt.ylabel("Declared Value Z-score", fontsize=13)
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ["Normal", "Anomaly"], title="Transaction Type", frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Diagnostic 6 — Price Ratio vs Ensemble Score
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
ax = sns.scatterplot(
    data=df,
    x="price_ratio",
    y="ensemble_score",
    hue="anomaly_label",
    palette=["#4C72B0", "#D62728"],
    alpha=0.70,
    s=60,
    edgecolor="black",
    linewidth=0.3,
)
plt.axhline(
    y=best_thr,
    color="black",
    linestyle="--",
    linewidth=2,
    label=f"Threshold = {best_thr:.3f}",
)
plt.axvline(
    x=1.0, color="gray", linestyle=":", linewidth=2, label="Expected Price Ratio"
)
plt.title(
    "Price Ratio vs Ensemble Anomaly Score", fontsize=18, fontweight="bold", pad=15
)
plt.xlabel("Price Ratio (Observed / Peer Median)", fontsize=13)
plt.ylabel("Ensemble Score", fontsize=13)
handles, _ = ax.get_legend_handles_labels()
ax.legend(
    handles,
    ["Normal", "Anomaly"],
    title="Transaction Type",
    frameon=True,
    loc="upper left",
)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Diagnostic 7 — Isolation Forest Permutation Importance (non-circular)
# ───────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))


def iso_forest_ap_scorer(estimator, X, y_true):
    raw_score = -estimator.decision_function(X)
    return average_precision_score(y_true, raw_score)


perm_result = permutation_importance(
    best_iso_forest,
    X_scaled[test_mask],
    y[test_mask],
    scoring=iso_forest_ap_scorer,
    n_repeats=20,
    random_state=RANDOM_SEED,
)

perm_importance = pd.Series(perm_result.importances_mean, index=FEATURES)
perm_std = pd.Series(perm_result.importances_std, index=FEATURES)

order = perm_importance.sort_values().index
perm_importance = perm_importance[order]
perm_std = perm_std[order]

ax = plt.gca()
ax.barh(
    perm_importance.index,
    perm_importance.values,
    xerr=perm_std.values,
    color=sns.color_palette("viridis", len(perm_importance)),
    edgecolor="black",
    capsize=4,
    error_kw={"elinewidth": 1.5, "ecolor": "black"},
)
for i, (val, std) in enumerate(zip(perm_importance.values, perm_std.values)):
    ax.annotate(
        f"{val:.4f} ± {std:.4f}",
        (val + std + 0.001, i),
        va="center",
        fontsize=9,
    )
plt.title(
    "Isolation Forest — Permutation Feature Importance\n(Drop in Test AP when feature is shuffled)",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Mean Decrease in Average Precision")
plt.ylabel("Feature")
sns.despine()
plt.tight_layout()
plt.show()
print(perm_result)

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Diagnostic 8 — Train vs Val vs Test Performance (overfitting check)
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
train_pred = (ensemble_score[train_mask] >= best_thr).astype(int)
val_pred = (ensemble_score[val_mask] >= best_thr).astype(int)
test_pred = (ensemble_score[test_mask] >= best_thr).astype(int)


def _metrics(y_true, y_pred, score):
    return {
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_true, score),
    }


train_metrics = _metrics(y[train_mask], train_pred, ensemble_score[train_mask])
val_metrics = _metrics(y[val_mask], val_pred, ensemble_score[val_mask])
test_metrics = _metrics(y[test_mask], test_pred, ensemble_score[test_mask])

metrics_df = pd.DataFrame(
    {
        "Metric": train_metrics.keys(),
        "Train": train_metrics.values(),
        "Val": val_metrics.values(),
        "Test": test_metrics.values(),
    }
)
metrics_long = metrics_df.melt(id_vars="Metric", var_name="Dataset", value_name="Score")

ax = sns.barplot(
    data=metrics_long,
    x="Metric",
    y="Score",
    hue="Dataset",
    palette="Set2",
    edgecolor="black",
    linewidth=1,
)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=9, padding=3)
plt.title(
    "Train vs Val vs Test Performance Comparison",
    fontsize=18,
    fontweight="bold",
    pad=15,
)
plt.xlabel("")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(axis="y", linestyle="--", alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

### Diagnostic 9 — Segment-Level Performance (Fairness/Blind-Spot Check)

Aggregate test-set metrics (Module 15) can mask uneven performance across segments.
This view computes precision and recall _per product category_ and _per country_
(test split only), checking whether any segment is a systematic blind spot before the
model is trusted operationally across the full population.


In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Diagnostic 9 — Segment-Level Performance (Product Category & Country)
# ───────────────────────────────────────────────────────────────────────────

test_df = df.loc[test_mask].copy()
test_df["y_true"] = y[test_mask]
test_df["y_pred"] = pred[test_mask]


def _segment_metrics(group_col):
    rows = []
    for seg, g in test_df.groupby(group_col):
        n = len(g)
        n_pos = int(g["y_true"].sum())
        prec = precision_score(g["y_true"], g["y_pred"], zero_division=0)
        rec = recall_score(g["y_true"], g["y_pred"], zero_division=0)
        rows.append(
            {
                group_col: seg,
                "n_transactions": n,
                "n_actual_fraud": n_pos,
                "precision": prec,
                "recall": rec,
            }
        )
    return pd.DataFrame(rows).sort_values("n_actual_fraud", ascending=False)


by_category = _segment_metrics("product_group")
by_country = _segment_metrics("country_origin")

print("=" * 70)
print("  SEGMENT-LEVEL PERFORMANCE — Product Category (Test Split)")
print("=" * 70)
print(by_category.to_string(index=False))

print("\n" + "=" * 70)
print("  SEGMENT-LEVEL PERFORMANCE — Country of Origin (Test Split)")
print("=" * 70)
print(by_country.to_string(index=False))

print(
    "\nNote: segments with n_actual_fraud = 0 show recall = 0/0 → reported as 0.0 "
    "(zero_division=0); this reflects no positive examples in that segment within "
    "this test split, not a genuine detection failure."
)

# ───────── Visualization: Recall by Category (the metric that matters most, F2) ─────────
fig, axes = plt.subplots(1, 2, figsize=(15, 15 * 9 / 16))

sns.barplot(
    data=by_category,
    y="product_group",
    x="recall",
    color="#1F77B4",
    edgecolor="black",
    ax=axes[0],
)
axes[0].set_title("Recall by Product Category", fontweight="bold")
axes[0].set_xlim(0, 1.05)
axes[0].set_xlabel("Recall")

sns.barplot(
    data=by_country,
    y="country_origin",
    x="recall",
    color="#D62728",
    edgecolor="black",
    ax=axes[1],
)
axes[1].set_title("Recall by Country of Origin", fontweight="bold")
axes[1].set_xlim(0, 1.05)
axes[1].set_xlabel("Recall")

plt.tight_layout()
plt.show()

## 18. Transaction-Level Explainability

Per the assignment brief's requirement that flagged transactions be explainable to a
non-technical business official, `explain_transaction()` decomposes any transaction's
ensemble score $E_i$ back into its three multiplicative components
($s_{\text{stat}}^{a^\star}$, $s_{\text{iso}}^{b^\star}$, $s_{\text{ratio}}^{c^\star}$,
Module 12) and reports each component's relative share of $E_i$, alongside plain-language
risk drivers derived from the same thresholds used for classification ($\zeta^\star$ from
Module 14, $\tau_{\text{rare}}$ from Module 9.5) — no new, undocumented cutoffs are
introduced at explanation time.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Transaction-level Explanation
# ═══════════════════════════════════════════════════════════════════════════

df["stat_score"] = stat_score
df["iso_score"] = iso_score
df["ratio_signal"] = ratio_signal


def explain_transaction(index: int):
    row = df.loc[index]

    components = {
        "Statistical Detector": row["stat_score"] ** a_opt,
        "Isolation Forest": row["iso_score"] ** b_opt,
        "Price Ratio Signal": row["ratio_signal"] ** c_opt,
    }
    total = sum(components.values())
    contribution = {k: 100 * v / total for k, v in components.items()}

    reasons = []
    if abs(row["robust_z"]) > Z_THRESHOLD:
        direction_word = "higher" if row["robust_z"] > 0 else "lower"
        reasons.append(
            f"Price is {row['price_ratio']:.2f}× peer median ({direction_word}, "
            f"{abs(row['robust_z']):.1f}σ beyond the derived Z_THRESHOLD={Z_THRESHOLD:.2f})"
        )
    if row["country_risk"] == 1:
        reasons.append("High-risk trade corridor")
    if row["importer_freq"] <= RARE_IMPORTER_THRESHOLD:
        reasons.append(
            f"Rare importer activity (frequency ≤ {RARE_IMPORTER_THRESHOLD:.0f}, "
            f"bottom 5% of training distribution)"
        )
    if row["ensemble_score"] >= best_thr:
        reasons.append("Hybrid ensemble exceeded decision threshold")

    print("=" * 75)
    print(f"Transaction ID        : {row['txn_id']}")
    print(f"Risk Score            : {row['ensemble_score']:.3f}")
    print(f"Risk Level            : {row['risk_level']}")
    print(f"Prediction            : {row['predicted_direction']}")
    print("=" * 75)
    print("\nModel Component Contribution")
    for k, v in sorted(contribution.items(), key=lambda x: x[1], reverse=True):
        print(f"{k:<25} {v:6.2f}%")
    print("\nTop Risk Drivers")
    for i, rsn in enumerate(reasons, 1):
        print(f"{i}. {rsn}")
    print("=" * 75)

    return contribution

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Example — explain the single highest-risk transaction
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
idx = df["ensemble_score"].idxmax()
contribution = explain_transaction(idx)

plot_df = pd.DataFrame(
    contribution.items(), columns=["Component", "Contribution"]
).sort_values("Contribution", ascending=True)

ax = sns.barplot(
    data=plot_df, x="Contribution", y="Component", palette="viridis", edgecolor="black"
)
for p in ax.patches:
    ax.annotate(
        f"{p.get_width():.1f}%",
        (p.get_width(), p.get_y() + p.get_height() / 2),
        xytext=(5, 0),
        textcoords="offset points",
        va="center",
        fontsize=11,
        fontweight="bold",
    )
plt.title("Hybrid Ensemble Component Contribution", fontsize=17, fontweight="bold")
plt.xlabel("Relative Contribution (%)")
plt.ylabel("")
sns.despine()
plt.tight_layout()
plt.show()

## 19. Business Intelligence Dashboards

These views aggregate individual transaction-level scores (Module 12) into
portfolio-level summaries a compliance or customs team would monitor operationally:
the highest-risk transactions by invoicing direction (BI1), the overall
flagged/not-flagged split (BI2), risk-score distribution by product category (BI3),
fraud rate by country of origin — which should visibly track the
`COUNTRY_RISK_MULTIPLIER` injected in Module 5.2 (BI4), the over- vs. under-invoicing
mix by category (BI5), and feature distributions split by true anomaly label,
providing a visual complement to Diagnostic 4's correlation-based redundancy check (BI6).


In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# BI 1 — Top 10 Over & Under Invoicing Transactions
# ───────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(15, 15 * 9 / 16), sharex=True)

# Top 10 OVER_INVOICING
top_over = (
    df[df["predicted_direction"] == "OVER_INVOICING"]
    .sort_values("ensemble_score", ascending=False)
    .head(10)
)

# Top 10 UNDER_INVOICING
top_under = (
    df[df["predicted_direction"] == "UNDER_INVOICING"]
    .sort_values("ensemble_score", ascending=False)
    .head(10)
)

# ───────── Left Plot: Over Invoicing ─────────
sns.barplot(
    data=top_over,
    y="txn_id",
    x="ensemble_score",
    color="#D62728",
    edgecolor="black",
    linewidth=0.8,
    ax=axes[0],
)

axes[0].set_title("Top 10 Over-Invoicing", fontsize=15, fontweight="bold")
axes[0].set_xlabel("Ensemble Score")
axes[0].set_ylabel("Transaction ID")
axes[0].set_xlim(0, 1.02)

for p in axes[0].patches:
    width = p.get_width()
    axes[0].annotate(
        f"{width:.3f}",
        (width, p.get_y() + p.get_height() / 2),
        xytext=(5, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=9,
    )

# ───────── Right Plot: Under Invoicing ─────────
sns.barplot(
    data=top_under,
    y="txn_id",
    x="ensemble_score",
    color="#1F77B4",
    edgecolor="black",
    linewidth=0.8,
    ax=axes[1],
)

axes[1].set_title("Top 10 Under-Invoicing", fontsize=15, fontweight="bold")
axes[1].set_xlabel("Ensemble Score")
axes[1].set_ylabel("")
axes[1].set_xlim(0, 1.02)

for p in axes[1].patches:
    width = p.get_width()
    axes[1].annotate(
        f"{width:.3f}",
        (width, p.get_y() + p.get_height() / 2),
        xytext=(5, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=9,
    )

fig.suptitle(
    "Top 10 Highest-Risk Transactions by Invoicing Type",
    fontsize=18,
    fontweight="bold",
)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# BI 2 — Risk Level Distribution
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
ax = sns.countplot(
    data=df,
    x="risk_level",
    order=["LOW", "HIGH"],
    palette=["#4C72B0", "#D62728"],
    edgecolor="black",
    linewidth=1.2,
    width=0.55,
)
plt.title("Risk Level Distribution", fontsize=18, fontweight="bold", pad=15)
plt.xlabel("Risk Level", fontsize=13)
plt.ylabel("Number of Transactions", fontsize=13)

total_n = len(df)
for p in ax.patches:
    count = int(p.get_height())
    percent = count / total_n * 100
    ax.annotate(
        f"{count:,}\n({percent:.1f}%)",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        xytext=(0, 6),
        textcoords="offset points",
    )
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# BI 3 — Risk Score Distribution by Product Group
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
ax = sns.boxplot(
    data=df,
    x="product_group",
    y="ensemble_score",
    palette="viridis",
    showfliers=False,
    linewidth=1.2,
)
sns.stripplot(
    data=df.sample(min(1200, len(df)), random_state=RANDOM_SEED),
    x="product_group",
    y="ensemble_score",
    color="black",
    alpha=0.20,
    size=2.5,
)
plt.title(
    "Distribution of Ensemble Risk Score by Product Category",
    fontsize=18,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Product Category", fontsize=13)
plt.ylabel("Ensemble Risk Score", fontsize=13)
plt.xticks(rotation=30, ha="right")
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# BI 4 — Fraud Rate by Country of Origin
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
country_fraud = (
    df.groupby("country_origin")["anomaly_label"]
    .mean()
    .mul(100)
    .sort_values()
    .reset_index(name="Fraud Rate")
)

ax = sns.barplot(
    data=country_fraud,
    x="Fraud Rate",
    y="country_origin",
    palette="rocket",
    edgecolor="black",
    linewidth=1,
)
for p in ax.patches:
    ax.annotate(
        f"{p.get_width():.1f}%",
        (p.get_width(), p.get_y() + p.get_height() / 2),
        xytext=(5, 0),
        textcoords="offset points",
        va="center",
        fontsize=10,
    )
plt.title("Fraud Rate by Country of Origin", fontsize=18, fontweight="bold", pad=15)
plt.xlabel("Fraud Rate (%)")
plt.ylabel("Country")
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# BI 5 — Over vs Under Invoicing Across Product Categories
# ───────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 15 * 9 / 16))
direction_df = (
    df[df["true_direction"] != "NORMAL"]
    .groupby(["product_group", "true_direction"])
    .size()
    .reset_index(name="Count")
)

ax = sns.barplot(
    data=direction_df,
    x="product_group",
    y="Count",
    hue="true_direction",
    palette={"OVER_INVOICING": "#D62728", "UNDER_INVOICING": "#1F77B4"},
    edgecolor="black",
)
plt.title(
    "Over vs Under Invoicing Across Product Categories",
    fontsize=18,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Product Category")
plt.ylabel("Number of Fraud Transactions")
plt.xticks(rotation=90)
plt.legend(title="Fraud Type")
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# BI 6 — Feature Distribution by Transaction Class
# ───────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 15 * 9 / 16), dpi=150)

plots = [
    ("price_ratio", "Price Ratio"),
    ("robust_z", "Robust Z-score"),
    ("value_zscore", "Value Z-score"),
    ("qty_zscore", "Quantity Z-score"),
]

for ax, (feature, title) in zip(axes.flat, plots):
    sns.violinplot(
        data=df,
        x="anomaly_label",
        y=feature,
        palette=["#4C72B0", "#D62728"],
        inner="quartile",
        cut=0,
        ax=ax,
    )
    ax.set_xticklabels(["Normal", "Anomaly"])
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(feature)

sns.despine()
plt.suptitle(
    "Feature Distribution by Transaction Class", fontsize=20, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 20. Interactive Manual Predictor (Demonstration Only)

This widget re-implements the scoring path of Modules 9–14 for a single,
manually-entered hypothetical transaction — using the **same** train-derived peer
statistics ($\widetilde p_g$, $\text{MAD}_g$), scaler, fitted Isolation Forest,
ensemble weights $(a^\star,b^\star,c^\star)$, and thresholds ($\theta^\star$,
$\zeta^\star$, $\tau_{\text{rare}}$) as the batch pipeline, so a reviewer can sanity-check
the model's behavior interactively without retraining anything. It is a demonstration
aid, not an independent detector.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  Interactive Widget — Manual Transaction Predictor
# ═══════════════════════════════════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, clear_output

product_widget = widgets.Dropdown(
    options=sorted(PRODUCT_CATEGORIES.keys()), description="Product"
)
country_widget = widgets.Dropdown(options=ALL_COUNTRIES, description="Country")
qty_widget = widgets.IntText(value=100, description="Quantity")
price_widget = widgets.FloatText(value=500, description="Unit Price")
importer_widget = widgets.IntText(value=10, description="Importer Freq")
predict_btn = widgets.Button(
    description="Predict", button_style="success", icon="check"
)
output = widgets.Output()

In [ ]:
def predict_transaction(_):
    with output:
        clear_output()

        product = product_widget.value
        quantity = qty_widget.value
        unit_price = price_widget.value
        country = country_widget.value
        importer = importer_widget.value
        hs_code = HS_CODE_MAP.get(product)

        if hs_code is None or hs_code not in peer_median_map:
            print("Invalid product / missing hs_code mapping")
            return

        peer = peer_median_map[hs_code]
        mad = peer_mad_map[hs_code]
        robust_z = (unit_price - peer) / (mad + 1e-9)
        abs_z = abs(robust_z)
        price_ratio = unit_price / peer
        log_ratio = np.log1p(abs(price_ratio - 1))
        declared_value = quantity * unit_price
        value_z = (declared_value - value_mean) / value_std
        qty_z = (np.log1p(quantity) - qty_mean) / qty_std
        country_risk = int(country in HIGH_RISK_COUNTRIES)

        sample = pd.DataFrame(
            {
                "robust_z": [robust_z],
                "value_zscore": [value_z],
                "qty_zscore": [qty_z],
                "log_importer_freq": [np.log1p(importer)],
                "country_risk": [country_risk],
            }
        )

        X = scaler.transform(sample)
        iso_raw = -best_iso_forest.decision_function(X)[0]

        iso = min_max_scale_with_bounds(iso_raw, ISO_RAW_MIN, ISO_RAW_MAX)
        stat = min_max_scale_with_bounds(abs_z, STAT_RAW_MIN, STAT_RAW_MAX)
        ratio = min_max_scale_with_bounds(log_ratio, RATIO_RAW_MIN, RATIO_RAW_MAX)

        score = np.clip(stat**a_opt * iso**b_opt * ratio**c_opt, 0, 1)
        risk = "HIGH RISK" if score >= best_thr else "LOW RISK"

        direction = "NORMAL"
        if robust_z > Z_THRESHOLD:
            direction = "OVER INVOICING"
        elif robust_z < -Z_THRESHOLD:
            direction = "UNDER INVOICING"

        print("=" * 70)
        print("TBML MANUAL DETECTION (Demo Only)")
        print("=" * 70)
        print(f"Risk Score        : {score:.3f}")
        print(f"Prediction        : {risk}")
        print(f"Direction         : {direction}")
        print(f"Price Ratio       : {price_ratio:.2f}")
        print(f"Robust Z-score    : {robust_z:.2f}")
        print("=" * 70)

        components = {
            "Statistical": stat**a_opt,
            "Isolation Forest": iso**b_opt,
            "Price Ratio": ratio**c_opt,
        }
        total = sum(components.values())
        explain_pct = {k: 100 * v / total for k, v in components.items()}
        plot_df = pd.DataFrame(
            explain_pct.items(), columns=["Component", "Contribution"]
        ).sort_values("Contribution")

        plt.figure(figsize=(8, 4))
        ax = sns.barplot(
            data=plot_df, x="Contribution", y="Component", palette="viridis"
        )
        for p in ax.patches:
            ax.annotate(
                f"{p.get_width():.1f}%",
                (p.get_width(), p.get_y() + p.get_height() / 2),
                xytext=(5, 0),
                textcoords="offset points",
                va="center",
            )
        plt.title("Detector Contribution")
        plt.show()

        print("\nBusiness Explanation")
        if abs_z > Z_THRESHOLD:
            direction_word = "above" if robust_z > 0 else "below"
            print(
                f"✔ Unit price {direction_word} peer median "
                f"({price_ratio:.2f}×, {abs_z:.1f}σ beyond Z_THRESHOLD={Z_THRESHOLD:.2f})."
            )
        if country_risk:
            print("✔ High-risk trade corridor.")
        if importer <= RARE_IMPORTER_THRESHOLD:
            print(f"✔ Rare importer activity (freq ≤ {RARE_IMPORTER_THRESHOLD:.0f}).")
        if (
            risk == "LOW RISK"
            and abs_z <= Z_THRESHOLD
            and not country_risk
            and importer > RARE_IMPORTER_THRESHOLD
        ):
            print("✔ No major suspicious indicators detected.")


predict_btn.on_click(predict_transaction)
display(
    widgets.VBox(
        [
            product_widget,
            country_widget,
            qty_widget,
            price_widget,
            importer_widget,
            predict_btn,
            output,
        ]
    )
)

## 21. Robustness Validation via Repeated Stratified Splits

### 21.1 The Problem With a Single Point Estimate

All results in Modules 15–16 come from **one** stratified 60/20/20 partition (seed 42).
With a 0.5% anomaly rate, the test split contains only $\approx$100 positive
examples — small enough that a different random partition could plausibly shift
precision by several points through sampling variance alone, even if the underlying
detector is unchanged. Reporting a single number without a variance estimate is
precisely the failure mode Nitsch (2017) identifies across the trade-misinvoicing
literature: _"quantitative findings are heavily dependent on the underlying
assumptions... making estimation results largely a matter of faith."_

### 21.2 Repeated-Split Estimator

For $R=5$ independent seeds $\{42, 7, 123, 2024, 99\}$, the **entire pipeline** —
partitioning, peer-statistic fitting, contamination selection, ensemble-weight
selection, and threshold selection — is re-run from scratch on each seed $k$,
producing test-set metrics $M_k$ (precision, recall, F1, F2, AUC). The reported
estimate is:

$$
\bar M = \frac{1}{R}\sum_{k=1}^{R} M_k, \qquad
s_M = \sqrt{\frac{1}{R-1}\sum_{k=1}^{R}\big(M_k - \bar M\big)^2}
$$

Critically, this is **not** merely re-evaluating one fixed model on different test
samples (which would only capture test-sampling noise). Because contamination,
ensemble weights, and $\theta^\star$ are _independently re-selected per split_ using
that split's own validation data, $s_M$ captures the **full end-to-end pipeline
variance**, including sensitivity of the hyperparameter-selection procedure itself to
which rows happen to fall in train/val/test.

### 21.3 Reading the Result

A metric with $s_M \ll \bar M$ (e.g. recall $1.000 \pm 0.000$, AUC $1.000 \pm 0.000$)
indicates the corresponding decision is essentially invariant to the partition — strong
evidence for that specific claim. A metric with $s_M$ a non-trivial fraction of $\bar M$
(e.g. precision $0.880 \pm 0.059$) indicates the single-split point estimate reported in
Module 15 should be read as one draw from a distribution, not as the model's exact
performance — the confidence band, not the point value, is the honest claim.

### 21.4 Weight Stability Confirms Module 12.4

Beyond performance metrics, the ensemble weights $(a^\star,b^\star,c^\star)$ themselves
are re-derived per split. If $b^\star_{\text{iso}}$ remains consistently near zero
across all five independent partitions, this upgrades Module 12.4's observation from
"true for one split" to "a robust, repeatable property of this synthetic ground
truth" — reinforcing, rather than contradicting, the decision to leave the weight
search unrestricted (no artificial floor).


In [ ]:
# %%
# ═══════════════════════════════════════════════════════════════════════════
#  Robustness Check — Repeated Stratified Splits (addresses Nitsch, 2017:
#  point-estimate results can be misleadingly precise; report mean ± std
#  across multiple independent train/val/test partitions of the same data)
# ═══════════════════════════════════════════════════════════════════════════

N_REPEATS = 5
REPEAT_SEEDS = [42, 7, 123, 2024, 99]

repeat_results = []

for seed in REPEAT_SEEDS:
    # 1) Re-split (same df, different stratified partition)
    _df, _train_mask, _val_mask, _test_mask = perform_train_val_test_split(
        df, y, val_size=VAL_SIZE, test_size=TEST_SIZE, random_state=seed
    )
    _train_df = _df[_df["split"] == "train"]

    # 2) Re-fit peer stats / scaler on this split's TRAIN only
    _peer_median_map = _train_df.groupby("hs_code")["unit_price"].median()
    _peer_mad_map = _train_df.groupby("hs_code")["unit_price"].apply(
        lambda x: (x - x.median()).abs().median()
    )
    _peer_median = _df["hs_code"].map(_peer_median_map)
    _peer_mad = _df["hs_code"].map(_peer_mad_map)
    _robust_z = (_df["unit_price"] - _peer_median) / (_peer_mad + 1e-9)

    _value_mean = _train_df["declared_value"].mean()
    _value_std = _train_df["declared_value"].std()
    _value_z = (_df["declared_value"] - _value_mean) / _value_std

    _log_qty = np.log1p(_df["quantity"])
    _qty_mean = np.log1p(_train_df["quantity"]).mean()
    _qty_std = np.log1p(_train_df["quantity"]).std()
    _qty_z = (_log_qty - _qty_mean) / _qty_std

    _freq_map = _train_df["importer_id"].value_counts()
    _importer_freq = _df["importer_id"].map(_freq_map).fillna(0)
    _log_importer_freq = np.log1p(_importer_freq)

    _feat_df = pd.DataFrame(
        {
            "robust_z": _robust_z,
            "value_zscore": _value_z,
            "qty_zscore": _qty_z,
            "log_importer_freq": _log_importer_freq,
            "country_risk": _df["country_risk"],
        }
    )

    _scaler = StandardScaler()
    _X = np.zeros((len(_df), len(FEATURES)))
    _X[_train_mask] = _scaler.fit_transform(_feat_df.loc[_train_mask].fillna(0))
    _X[_val_mask] = _scaler.transform(_feat_df.loc[_val_mask].fillna(0))
    _X[_test_mask] = _scaler.transform(_feat_df.loc[_test_mask].fillna(0))

    # 3) Re-select IsolationForest contamination on VAL
    _best_ap_iso, _best_iso = -1.0, None
    _iso_score = None
    for c_val in CONTAMINATION_GRID:
        _forest = IsolationForest(
            n_estimators=300,
            contamination=c_val,
            max_samples="auto",
            random_state=seed,
            n_jobs=-1,
        )
        _forest.fit(_X[_train_mask])
        _cand = min_max_scale(-_forest.decision_function(_X))
        _ap = average_precision_score(y[_val_mask], _cand[_val_mask])
        if _ap > _best_ap_iso:
            _best_ap_iso, _best_iso, _iso_score = _ap, _forest, _cand

    _stat_score = min_max_scale(_robust_z.abs().values)
    _price_ratio = _df["unit_price"] / (_peer_median + 1e-9)
    _log_ratio = np.log1p(np.abs(_price_ratio - 1))
    _ratio_signal = min_max_scale(_log_ratio.values)

    # 4) Re-select ensemble weights on VAL
    _best_ap, _best_w, _ens_score = -1.0, None, None
    for a in np.arange(0.0, 1.0 + WEIGHT_STEP, WEIGHT_STEP):
        for b in np.arange(0.0, 1.0 - a + WEIGHT_STEP, WEIGHT_STEP):
            c = round(1.0 - a - b, 4)
            if c < 0:
                continue
            _cand = min_max_scale(
                (_stat_score**a) * (_iso_score**b) * (_ratio_signal**c)
            )
            _ap = average_precision_score(y[_val_mask], _cand[_val_mask])
            if _ap > _best_ap:
                _best_ap, _best_w, _ens_score = _ap, (a, b, c), _cand

    # 5) Re-select F2-optimal threshold on VAL
    _best_f2, _best_t = 0.0, 0.0
    for pct in np.arange(85.0, 99.5, 0.1):
        thr = np.percentile(_ens_score[_val_mask], pct)
        _pv = (_ens_score[_val_mask] >= thr).astype(int)
        f2 = fbeta_score(y[_val_mask], _pv, beta=2, zero_division=0)
        if f2 > _best_f2:
            _best_f2, _best_t = f2, thr

    _pred = (_ens_score >= _best_t).astype(int)

    repeat_results.append(
        {
            "seed": seed,
            "a_stat": _best_w[0],
            "b_iso": _best_w[1],
            "c_ratio": _best_w[2],
            "precision": precision_score(
                y[_test_mask], _pred[_test_mask], zero_division=0
            ),
            "recall": recall_score(y[_test_mask], _pred[_test_mask], zero_division=0),
            "f1": f1_score(y[_test_mask], _pred[_test_mask], zero_division=0),
            "f2": fbeta_score(
                y[_test_mask], _pred[_test_mask], beta=2, zero_division=0
            ),
            "auc": roc_auc_score(y[_test_mask], _ens_score[_test_mask]),
        }
    )

repeat_df = pd.DataFrame(repeat_results)

print("=" * 70)
print(f"  ROBUSTNESS CHECK — {N_REPEATS} independent train/val/test splits")
print("=" * 70)
print(repeat_df.to_string(index=False))
print("-" * 70)
summary = repeat_df.drop(columns="seed").agg(["mean", "std"]).T
summary.columns = ["Mean", "Std Dev"]
print(summary.round(4).to_string())
print("=" * 70)
print(
    "\nNote: metrics vary across splits because IsolationForest contamination, "
    "ensemble weights, and decision threshold are independently re-tuned per "
    "split (not just re-evaluated) — this reflects true end-to-end pipeline "
    "variance, not just sampling noise in the test set."
)
print("\n" + "=" * 70)
print("  ENSEMBLE WEIGHT STABILITY ACROSS SPLITS")
print("=" * 70)
print(repeat_df[["seed", "a_stat", "b_iso", "c_ratio"]].to_string(index=False))
print("-" * 70)
weight_summary = repeat_df[["a_stat", "b_iso", "c_ratio"]].agg(["mean", "std"]).T
weight_summary.columns = ["Mean", "Std Dev"]
print(weight_summary.round(4).to_string())
print(
    "\nInterpretation: if b_iso stays near 0 across all 5 independently re-tuned "
    "splits, this confirms the finding in Module 12.4 is a stable property of this "
    "synthetic dataset's price-dominated ground truth — not an artifact of one "
    "particular train/val/test partition."
)

## 22. Limitations & Assumptions

This section consolidates every simplifying assumption made across Modules 5–21,
following Nitsch (2017)'s core methodological principle: quantitative claims about
trade mis-invoicing detection are only meaningful when their generating assumptions
are stated explicitly alongside them.

### 22.1 Single-Source Detection (No Partner Country Method)

Choi (2019)'s methodology derives much of its false-positive reduction from
**cross-referencing two independent data sources** — the Price Filter Method (PFM,
unit-price analysis) and the Partner Country Method (PCM, mirror trade-statistics
analysis). This project implements only a PFM-style analysis (Module 9.2, 10) plus a
multivariate detector (Module 11); no mirror/partner-country data exists in the
synthetic single-source setting. The hybrid ensemble's requirement that multiple
_internally derived_ signals agree (Module 12.1) is a partial analogue, but it is not
equivalent to genuine independent-source verification, and real deployment would
likely see a materially higher false-positive rate without a PCM-style second source.

### 22.2 Synthetic Ground Truth Is More Separable Than Reality

The injected price-multiplier bands (Module 5.4, 3–8× / 0.05–0.30×) are deliberately
conservative relative to documented real cases (up to ~2400× per Simmons & Simmons),
yet still produce near-perfect AUC ($1.000 \pm 0.000$ across all five robustness
splits, Module 21). This indicates the synthetic anomalies remain more separable in
feature space than real-world mis-invoicing, which — per both Choi (2019) and Nitsch
(2017) — is frequently confounded with legitimate explanations: CIF/FOB valuation
differences, HS-code misclassification, currency and timing effects, and genuine
quality-tier price variation within a product category (e.g. smartphones legitimately
ranging \$100–\$800). None of these confounders are modeled here; a production system
would need to account for them explicitly before the reported lift and precision
figures could be expected to hold.

### 22.3 No Homogeneous-Group Refinement (Clustering)

Choi (2019, Section 7) found that further sub-clustering each HS-code group by
unit price, weight-price, and distance (via k-means) sometimes improves and sometimes
_worsens_ targeting accuracy, depending on whether the arbitrary cluster count matches
the true heterogeneity of that product category. This project uses only the HS-code
grouping (Module 9.2) without such refinement; given Choi's mixed results, this is a
defensible simplification rather than an oversight, but remains a direction for
product-specific future work.

### 22.4 Distance, Freight, and Currency Are Not Modeled

Both Choi (2019) and Nitsch (2017) list distance-driven freight cost (CIF vs. FOB) and
currency/exchange-rate effects among the most common _legitimate_ causes of price and
mirror-statistic discrepancies. Neither is present in the synthetic generator (Module
5), meaning the model cannot presently distinguish "genuinely far away, hence
legitimately more expensive" from "over-invoiced" — a gap that would need to be closed
before deployment on real customs data.

### 22.5 Sequential, Not Joint, Hyperparameter Selection

Contamination (Module 11), ensemble weights (Module 12), and the decision threshold
(Module 13) are selected **sequentially**, each holding the previous choice fixed. A
jointly-optimal combination might differ from this greedy sequence's result. This is a
standard simplification in applied pipelines but is noted explicitly as a
simplification rather than a claim of global optimality.

### 22.6 Single Country/Importer Risk Model

The country-risk and shell-importer multipliers (Module 5.2) are constant across all
product categories and time. Real risk is almost certainly product- and time-varying
(e.g. Kellenberg & Levinson's finding that tariff rate and corruption jointly predict
trade gaps) — a refinement not attempted here.

### 22.7 Summary Stance

None of the above limitations invalidate the pipeline's internal rigor (leakage-safe
statistics, validation-only hyperparameter selection, repeated-split confidence
intervals) — they bound the **external validity** of the specific numeric results
(precision, lift) to this synthetic dataset's generating assumptions, exactly as
Section 21 argues a single point estimate should never be read without its confidence
band, and exactly as Nitsch (2017) argues no trade-misinvoicing estimate should be read
without its assumptions.
